In [19]:
print("hi")

hi


# Complete Analysis -- Hindi Verbal Fluency & SpAM (Phase 2+)

This notebook performs an exhaustive analysis of the Hindi VFT + SpAM dataset
covering 12 research questions, plus EDA blocks. Designed to run end-to-end on
Google Colab. Embedding analysis (RQ6) benefits from a GPU runtime but works on CPU.

**Original RQs (kept):**
- RQ1: Phonological similarity trend over retrieval position
- RQ2: Joint cue model -- SpAM + Phonology -> IRT
- RQ3: Domain differences in clustering
- RQ4: Fluency x domain x position GLM (Hi_Read & Hi_Write decomposed)

**New RQs:**
- RQ5: Script choice & Hinglish penalty
- RQ6: Embedding-based semantic distance (MiniLM, LaBSE, MuRIL)
- RQ7: Patch foraging (Marginal Value Theorem)
- RQ8: Word typicality / production frequency
- RQ9: Individual-difference battery (L1, decoupled fluencies, chronotype, gender)
- RQ10: Cross-domain trait consistency
- RQ11: SpAM <-> VFT order alignment
- RQ12: Strategy text mining (exploratory)


## Section 0 -- Setup

In [ ]:
# # -*- coding: utf-8 -*-
# from google.colab import drive
# drive.mount('/content/drive')


In [4]:
!pip install -q indic-transliteration sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 3.9 MB/s eta 0:00:00


In [5]:
import json, os, re, pickle, warnings, sys
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist, cdist
from scipy.stats import (pearsonr, spearmanr, mannwhitneyu, kruskal,
                         f_oneway, wilcoxon, kendalltau, chi2_contingency)

import statsmodels.formula.api as smf
import statsmodels.api as sm

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#333333', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B',
           '#588B8B', '#FFD166', '#06AED5']
OUTPUT_DIR = '/content/phase2_outputs_complete'
DATAFRAMES_DIR = f'{OUTPUT_DIR}/dataframes'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATAFRAMES_DIR, exist_ok=True)
EMB_CACHE_PATH = f'{OUTPUT_DIR}/embeddings_cache.pkl'

print(f'Output directory: {OUTPUT_DIR}')


Output directory: /content/phase2_outputs_complete


## Section 1 -- Data loading & extended preprocessing

In [6]:
print('=' * 70)
print('DATA LOADING')
print('=' * 70)

DATA_PATH = '/content/BRSM/responses.json'
if not os.path.exists(DATA_PATH):
    alt = '/content/BRSM/responses (1).json'
    if os.path.exists(alt):
        DATA_PATH = alt
if not os.path.exists(DATA_PATH):
    # Local upload fallback
    cand = ['/content/responses.json', '/content/responses (1).json']
    for c in cand:
        if os.path.exists(c):
            DATA_PATH = c; break
print(f'Loading: {DATA_PATH}')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data['fluency-spam'], orient='index').reset_index(drop=True)
print(f'Subjects in raw JSON: {len(df)}')


DATA LOADING
Loading: /content/responses (1).json
Subjects in raw JSON: 35


In [7]:
# ---- Parse VFT trials ----
vft_results = []
for _, row in df.iterrows():
    sid = row['subject_id']
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        if trial.get('task') == 'VFT' and trial.get('trial_type') == 'html-keyboard-response':
            domain = trial.get('domain', 'unknown')
            if 'practice' in domain.lower():
                continue
            tagged = trial.get('tagged_responses', [])
            if isinstance(tagged, str):
                try: tagged = json.loads(tagged)
                except: tagged = []
            rts = trial.get('response_times', [])
            if isinstance(rts, str):
                try: rts = json.loads(rts)
                except: rts = []
            words = [item.get('response', '') for item in tagged
                     if isinstance(item, dict) and 'response' in item]
            vft_results.append({
                'subject_id': sid, 'domain': domain,
                'word_count': len(words),
                'mean_irt': np.mean(rts) if rts else np.nan,
                'words': words, 'irts': rts
            })
df_vft = pd.DataFrame(vft_results)

# ---- Parse SpAM trials ----
spam_results = []
for _, row in df.iterrows():
    sid = row['subject_id']
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        if trial.get('task') == 'SpAM':
            domain = trial.get('domain', 'unknown')
            if 'practice' in domain.lower():
                continue
            dw = trial.get('droppedwords', [])
            if isinstance(dw, str):
                try: dw = json.loads(dw)
                except: dw = []
            coords_dict = {}
            for item in dw:
                if isinstance(item, dict):
                    w = item.get('word')
                    x = item.get('x_norm'); y = item.get('y_norm')
                    if w is not None and x is not None and y is not None:
                        coords_dict[w] = {'word': w, 'x': float(x), 'y': float(y)}
            word_coords = list(coords_dict.values())
            pairwise = []
            if len(word_coords) >= 2:
                coords = [[wc['x'], wc['y']] for wc in word_coords]
                wds = [wc['word'] for wc in word_coords]
                dists = pdist(coords, metric='euclidean')
                k = 0
                for i in range(len(wds)):
                    for j in range(i + 1, len(wds)):
                        pairwise.append({'word1': wds[i], 'word2': wds[j],
                                         'distance': dists[k]})
                        k += 1
            spam_results.append({
                'subject_id': sid, 'domain': domain,
                'word_count': len(word_coords),
                'word_coords': word_coords,
                'pairwise_distances': pairwise,
                'viewport_w': trial.get('viewport_width'),
                'viewport_h': trial.get('viewport_height'),
                'device_pixel_ratio': trial.get('device_pixel_ratio'),
            })
df_spam = pd.DataFrame(spam_results)

print(f'VFT trials (non-practice): {len(df_vft)}')
print(f'SpAM trials (non-practice): {len(df_spam)}')
print(f'Domains in VFT: {df_vft["domain"].value_counts().to_dict()}')


VFT trials (non-practice): 105
SpAM trials (non-practice): 105
Domains in VFT: {'animals': 35, 'foods': 35, 'body-parts': 24, 'colours': 11}


In [8]:
# ---- Build df_subjects with FULL demographics ----
SUBJECT_FIELDS = [
    'Hi_Read', 'Hi_Write', 'En_Read', 'En_Write',
    'age', 'gender', 'first_language', 'state_ut', 'education',
    'dominant_hand', 'alert_time',
    'language_count', 'languages_list', 'language_acquisition',
    'strategies', 'other_info'
]

subject_rows = []
for _, row in df.iterrows():
    sid = row['subject_id']
    entry = {'subject_id': sid}
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        ttype = trial.get('trial_type', '')
        if ttype in ('survey-html-form', 'survey-multi-choice', 'survey-text'):
            for key in SUBJECT_FIELDS:
                if key in trial and trial[key] not in (None, ''):
                    # Don't overwrite a previously-set value
                    if entry.get(key) in (None, ''):
                        entry[key] = trial[key]
    subject_rows.append(entry)

df_subjects = pd.DataFrame(subject_rows).drop_duplicates(subset=['subject_id']).reset_index(drop=True)

# Numeric coercions
for col in ['Hi_Read', 'Hi_Write', 'En_Read', 'En_Write', 'age',
            'education', 'language_count']:
    if col in df_subjects.columns:
        df_subjects[col] = pd.to_numeric(df_subjects[col], errors='coerce')

# Convenience aggregates
hi_cols = [c for c in ['Hi_Read', 'Hi_Write'] if c in df_subjects.columns]
en_cols = [c for c in ['En_Read', 'En_Write'] if c in df_subjects.columns]
df_subjects['hi_fluency'] = df_subjects[hi_cols].mean(axis=1) if hi_cols else np.nan
df_subjects['en_fluency'] = df_subjects[en_cols].mean(axis=1) if en_cols else np.nan

# L1 binary: Hindi vs other
def _l1_binary(v):
    if not isinstance(v, str): return np.nan
    return 'Hindi' if 'hindi' in v.lower() else 'Other'
df_subjects['l1_hindi'] = df_subjects['first_language'].apply(_l1_binary) if 'first_language' in df_subjects.columns else np.nan

# Gender normalisation (handle case differences)
if 'gender' in df_subjects.columns:
    df_subjects['gender_norm'] = df_subjects['gender'].astype(str).str.strip().str.lower().replace(
        {'m':'M','male':'M','f':'F','female':'F','nan':np.nan,'none':np.nan})

print(f'Subjects with demographics: {len(df_subjects)}')
print(f'Columns: {list(df_subjects.columns)}')
print(df_subjects[['subject_id','age','gender_norm','first_language','l1_hindi','Hi_Read','Hi_Write','En_Read','En_Write']].head().to_string())


Subjects with demographics: 35
Columns: ['subject_id', 'Hi_Read', 'Hi_Write', 'En_Read', 'En_Write', 'first_language', 'language_count', 'languages_list', 'language_acquisition', 'state_ut', 'age', 'gender', 'education', 'dominant_hand', 'alert_time', 'other_info', 'strategies', 'hi_fluency', 'en_fluency', 'l1_hindi', 'gender_norm']
   subject_id  age gender_norm first_language l1_hindi  Hi_Read  Hi_Write  En_Read  En_Write
0       10255   27           M         Telugu    Other        4         4        5         5
1       95712   23           M          Hindi    Hindi        5         4        5         4
2       53105   21           M       Gujarati    Other        5         5        5         5
3       53307   21           M        Punjabi    Other        5         3        5         5
4       92821   25           M          Hindi    Hindi        4         4        4         4


## Section 2 -- Helper functions

In [9]:
# Script + edit-distance helpers
def is_devanagari(word):
    if not isinstance(word, str):
        return False
    return bool(re.search(r'[ऀ-ॿ]', word))

def has_latin(word):
    if not isinstance(word, str):
        return False
    return bool(re.search(r'[A-Za-z]', word))

def romanize(word):
    if not isinstance(word, str):
        return word
    if is_devanagari(word):
        return transliterate(word, sanscript.DEVANAGARI, sanscript.ITRANS).lower()
    return word.lower()

def normalized_edit_distance(a, b):
    a, b = romanize(a), romanize(b)
    if not a or not b:
        return np.nan
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return 1 - dp[m][n] / max(m, n)


In [10]:
# Hinglish detector: per-domain English shortlists.
# A Latin-script word is classified as 'english' if it appears (case-insensitive,
# stripped of plural/whitespace) in the domain shortlist; otherwise 'hinglish'.

ENGLISH_VOCAB = {
    'animals': {
        'dog','cat','lion','tiger','elephant','horse','cow','goat','sheep','pig',
        'monkey','rabbit','rat','mouse','snake','crocodile','bear','wolf','fox',
        'deer','buffalo','donkey','camel','giraffe','zebra','panda','kangaroo',
        'leopard','cheetah','bull','ox','calf','puppy','kitten','squirrel','frog',
        'lizard','crow','sparrow','parrot','peacock','pigeon','duck','hen','chicken',
        'eagle','owl','vulture','swan','penguin','dolphin','whale','shark','fish',
        'turtle','tortoise','octopus','crab','lobster','butterfly','bee','ant',
        'spider','scorpion','snail','worm','fly','mosquito','bat','hippo','rhino',
        'jackal','hyena','antelope','gorilla','chimp','chimpanzee','rooster',
        'goose','goat','calf','foal','jaguar','puma','cougar','seal','walrus',
    },
    'foods': {
        'rice','bread','roti','chapati','dal','curry','samosa','pakora','biryani',
        'dosa','idli','vada','poha','upma','paratha','naan','khichdi','rajma',
        'chana','chhole','chole','paneer','butter','milk','curd','yogurt','ghee',
        'sugar','salt','chilli','chili','pepper','onion','garlic','ginger',
        'tomato','potato','carrot','cucumber','spinach','cabbage','cauliflower',
        'pumpkin','bottle gourd','okra','ladyfinger','brinjal','eggplant','peas',
        'beans','lentil','wheat','corn','maize','barley','oats','egg','meat',
        'chicken','mutton','fish','prawn','apple','banana','mango','orange',
        'grape','pomegranate','watermelon','pineapple','papaya','guava','lemon',
        'lime','strawberry','tea','coffee','water','juice','lassi','milkshake',
        'cake','cookie','biscuit','chocolate','icecream','ice cream','pizza',
        'pasta','burger','sandwich','noodles','soup','salad','popcorn','chips',
        'pickle','jam','honey','flour','semolina','rava','gulab jamun','jalebi',
        'rasgulla','laddu','kheer','halwa','sweets','sweet','sambar','rasam',
        'vada','poori','puri','bhature','aloo','sabzi','curry','masala',
    },
    'body-parts': {
        'hand','arm','leg','foot','feet','head','hair','eye','eyes','ear','ears',
        'nose','mouth','tongue','tooth','teeth','lip','lips','cheek','chin',
        'forehead','neck','shoulder','elbow','wrist','finger','thumb','nail',
        'palm','knee','thigh','ankle','heel','toe','toes','chest','back','belly',
        'stomach','waist','hip','hips','navel','heart','lung','lungs','liver',
        'kidney','brain','skin','bone','muscle','blood','vein','spine','rib',
        'ribs','jaw','eyebrow','eyelash','eyelid','nostril','throat','beard',
        'moustache','mustache','calf','shin','knuckle','wrist','breast','nail',
    },
    'colours': {
        'red','blue','green','yellow','orange','purple','pink','black','white',
        'grey','gray','brown','violet','indigo','magenta','cyan','maroon','beige',
        'cream','gold','silver','bronze','copper','navy','teal','olive','turquoise',
        'lavender','peach','salmon','crimson','scarlet','aqua','tan','khaki',
        'mustard','rust','wine','rose','coral','mint','ivory','pearl','platinum',
        'amber','jade','emerald','ruby','sapphire','azure','lime','plum','orchid',
        'saffron','fuchsia','chartreuse','periwinkle','vermilion','sienna','ochre',
    },
}

# Heuristic Hinglish-suffix list (transliterated Hindi often ends like this)
HINGLISH_SUFFIXES = ('aa','ii','ee','ay','iya','iyaan','ein','aon','wala','vala')

def classify_word(word, domain):
    '''Return one of: pure_devanagari, pure_english, hinglish, mixed_or_other.'''
    if not isinstance(word, str) or not word.strip():
        return 'unknown'
    w = word.strip().lower()
    has_dev = is_devanagari(w)
    has_lat = has_latin(w)
    if has_dev and not has_lat:
        return 'pure_devanagari'
    if has_dev and has_lat:
        return 'mixed_or_other'
    if has_lat:
        vocab = ENGLISH_VOCAB.get(domain, set())
        # strip trailing s for crude singular check
        w_singular = w.rstrip('s') if w.endswith('s') and len(w) > 3 else w
        if w in vocab or w_singular in vocab:
            return 'pure_english'
        # heuristic: ends with Hindi-style suffix
        if any(w.endswith(suf) for suf in HINGLISH_SUFFIXES):
            return 'hinglish'
        # short common english words not in our list -> still call english if very short and lacks Hindi suffix
        if len(w) <= 4:
            return 'pure_english'
        # default for medium-length latin word not in our vocab: hinglish
        return 'hinglish'
    return 'unknown'

# Sanity
for w, d in [('dog','animals'), ('कुत्ता','animals'), ('kutta','animals'),
             ('hand','body-parts'), ('हाथ','body-parts'), ('haath','body-parts'),
             ('red','colours'), ('लाल','colours'), ('lal','colours'),
             ('rice','foods'), ('चावल','foods'), ('chawal','foods')]:
    print(f"  {d:12s} {w!r:18s} -> {classify_word(w, d)}")


  animals      'dog'              -> pure_english
  animals      'कुत्ता'           -> pure_devanagari
  animals      'kutta'            -> hinglish
  body-parts   'hand'             -> pure_english
  body-parts   'हाथ'              -> pure_devanagari
  body-parts   'haath'            -> hinglish
  colours      'red'              -> pure_english
  colours      'लाल'              -> pure_devanagari
  colours      'lal'              -> pure_english
  foods        'rice'             -> pure_english
  foods        'चावल'             -> pure_devanagari
  foods        'chawal'           -> hinglish


In [11]:
# MVT (Marginal Value Theorem) switch detector
def mvt_switches(irts):
    '''Given a list of IRTs for one trial, return a list of 0/1 switch flags
    (length len(irts)-1). A switch is flagged at position i (between word i and i+1)
    when the local rate (1/irts[i+1]) falls below the long-run mean rate up to that point.
    Hills, Jones & Todd (2012) operationalisation.'''
    irts = [r for r in irts if isinstance(r, (int, float)) and r > 0]
    n = len(irts)
    if n < 2:
        return []
    flags = []
    for i in range(1, n):
        cum_time = sum(irts[:i+1])
        global_rate = (i + 1) / cum_time          # words per ms so far
        local_rate  = 1 / irts[i]                  # rate at this step
        flags.append(1 if local_rate < global_rate else 0)
    return flags


In [12]:
# Embedding loader with caching
def load_or_compute_embeddings(unique_words, cache_path=EMB_CACHE_PATH):
    '''Returns dict: model_name -> {word -> np.array}.
    Loads cached embeddings if available; otherwise computes & saves.'''
    cache = {}
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'rb') as f:
                cache = pickle.load(f)
            print(f'  Loaded cache: {sum(len(v) for v in cache.values())} entries across {len(cache)} models')
        except Exception as e:
            print(f'  Cache load failed: {e} -- recomputing')
            cache = {}

    needed_models = ['paraphrase-multilingual-MiniLM-L12-v2',
                     'sentence-transformers/LaBSE',
                     'google/muril-base-cased']

    # Determine which models / words need computing
    to_compute = {}
    for m in needed_models:
        existing = cache.get(m, {})
        missing = [w for w in unique_words if w not in existing]
        if missing:
            to_compute[m] = missing

    if not to_compute:
        print('  All embeddings already cached.')
        return cache

    # Imports happen lazily so first-cell imports stay light
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'  Embedding device: {device}')

    for model_name, words in to_compute.items():
        print(f'\\n  Computing {len(words)} embeddings with {model_name}')
        if model_name.startswith('google/muril'):
            from transformers import AutoTokenizer, AutoModel
            tok = AutoTokenizer.from_pretrained(model_name)
            mdl = AutoModel.from_pretrained(model_name).to(device).eval()
            embs = {}
            with torch.no_grad():
                B = 32
                for s in range(0, len(words), B):
                    batch = words[s:s+B]
                    enc = tok(batch, padding=True, truncation=True, return_tensors='pt').to(device)
                    out = mdl(**enc).last_hidden_state          # (B, T, H)
                    mask = enc.attention_mask.unsqueeze(-1).float()
                    pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
                    pooled = pooled.cpu().numpy()
                    for w, v in zip(batch, pooled):
                        embs[w] = v
        else:
            from sentence_transformers import SentenceTransformer
            mdl = SentenceTransformer(model_name, device=device)
            vecs = mdl.encode(words, batch_size=64, show_progress_bar=False,
                              convert_to_numpy=True, normalize_embeddings=False)
            embs = {w: v for w, v in zip(words, vecs)}

        cache.setdefault(model_name, {}).update(embs)

        # Persist after each model so we can resume on failure
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump(cache, f)
        except Exception as e:
            print(f'  WARNING: cache write failed: {e}')

    return cache

def cosine_dist(u, v):
    if u is None or v is None:
        return np.nan
    nu = np.linalg.norm(u); nv = np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return np.nan
    return 1.0 - float(np.dot(u, v) / (nu * nv))


## Section 3 -- Build transition / word / cluster dataframes

In [13]:
print('=' * 70)
print('BUILDING DATAFRAMES')
print('=' * 70)

transitions = []
word_level_rows = []

for _, vft_row in df_vft.iterrows():
    subj, domain = vft_row['subject_id'], vft_row['domain']
    words, irts = vft_row['words'], vft_row['irts']

    spam_match = df_spam[(df_spam['subject_id'] == subj) & (df_spam['domain'] == domain)]
    if spam_match.empty:
        continue

    dist_lookup = {}
    for item in spam_match.iloc[0]['pairwise_distances']:
        dist_lookup[(item['word1'], item['word2'])] = item['distance']
        dist_lookup[(item['word2'], item['word1'])] = item['distance']

    coord_lookup = {wc['word']: (wc['x'], wc['y'])
                    for wc in spam_match.iloc[0]['word_coords']}

    all_dists = [item['distance'] for item in spam_match.iloc[0]['pairwise_distances']]
    trial_mean = np.mean(all_dists) if all_dists else 0
    trial_std  = np.std(all_dists) if all_dists else 1
    if trial_std == 0: trial_std = 1
    trial_median = np.median(all_dists) if all_dists else 0.5

    # MVT switches
    mvt_flags = mvt_switches(irts)

    min_len = min(len(words), len(irts))
    for i in range(1, min_len):
        w_prev, w_curr = words[i-1], words[i]
        irt = irts[i]
        spam_dist = dist_lookup.get((w_prev, w_curr), np.nan)
        spam_dist_z = (spam_dist - trial_mean) / trial_std if not np.isnan(spam_dist) else np.nan
        phon_sim = normalized_edit_distance(w_prev, w_curr)
        is_switch_median = int(spam_dist > trial_median) if not np.isnan(spam_dist) else np.nan
        is_switch_mean   = int(spam_dist > trial_mean)   if not np.isnan(spam_dist) else np.nan
        mvt_switch = mvt_flags[i-1] if (i-1) < len(mvt_flags) else np.nan

        transitions.append({
            'subject_id': subj, 'domain': domain,
            'position': i + 1,
            'word_prev': w_prev, 'word_curr': w_curr,
            'cls_prev': classify_word(w_prev, domain),
            'cls_curr': classify_word(w_curr, domain),
            'irt_ms': irt,
            'log_irt': np.log1p(irt) if irt > 0 else np.nan,
            'spam_dist': spam_dist, 'spam_dist_z': spam_dist_z,
            'phon_sim': phon_sim,
            'is_switch': is_switch_median,
            'is_switch_mean': is_switch_mean,
            'is_switch_mvt': mvt_switch,
            'trial_word_count': min_len,
        })

    for i in range(min_len):
        word = words[i]; irt_val = irts[i]
        if word in coord_lookup:
            dists_other = [dist_lookup.get((word, other), np.nan)
                           for other in coord_lookup if other != word]
            dists_other = [d for d in dists_other if not np.isnan(d)]
            mean_neigh = np.mean(dists_other) if dists_other else np.nan
            mean_neigh_z = (mean_neigh - trial_mean) / trial_std if not np.isnan(mean_neigh) else np.nan
        else:
            mean_neigh = np.nan; mean_neigh_z = np.nan

        word_level_rows.append({
            'subject_id': subj, 'domain': domain,
            'position': i + 1, 'word': word,
            'cls': classify_word(word, domain),
            'irt_ms': irt_val,
            'log_irt': np.log1p(irt_val) if irt_val > 0 else np.nan,
            'mean_neigh_dist': mean_neigh,
            'mean_neigh_dist_z': mean_neigh_z,
        })

df_trans = pd.DataFrame(transitions)
df_words = pd.DataFrame(word_level_rows)

# Filter outliers
df_trans = df_trans[(df_trans['irt_ms'] > 0) & (df_trans['irt_ms'] < 30000)].dropna(
    subset=['spam_dist','phon_sim','log_irt']).reset_index(drop=True)
df_words = df_words[(df_words['irt_ms'] > 0) & (df_words['irt_ms'] < 30000)].dropna(
    subset=['log_irt']).reset_index(drop=True)

# Merge subject demographics
demog_cols = ['subject_id','Hi_Read','Hi_Write','En_Read','En_Write','hi_fluency','en_fluency',
              'l1_hindi','first_language','age','gender_norm','education','dominant_hand',
              'alert_time','language_count','strategies']
demog_cols = [c for c in demog_cols if c in df_subjects.columns]
df_trans = df_trans.merge(df_subjects[demog_cols], on='subject_id', how='left')
df_words = df_words.merge(df_subjects[demog_cols], on='subject_id', how='left')

# Position scaled within trial
df_trans['position_scaled'] = df_trans.groupby(['subject_id','domain'])['position'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0.5)

print(f'Transition rows: {len(df_trans)}')
print(f'Word rows: {len(df_words)}')
print(f'Domains: {df_trans["domain"].value_counts().to_dict()}')


BUILDING DATAFRAMES
Transition rows: 888
Word rows: 1038
Domains: {'animals': 310, 'foods': 275, 'body-parts': 167, 'colours': 136}


In [14]:
# Cluster-level frame using SpAM-median switch
cluster_rows = []
for (subj, domain), grp in df_trans.groupby(['subject_id','domain']):
    sw = grp['is_switch'].values
    sizes = []; cur = 1
    for s in sw:
        if s == 0: cur += 1
        else: sizes.append(cur); cur = 1
    sizes.append(cur)
    cluster_rows.append({
        'subject_id': subj, 'domain': domain,
        'n_words': len(grp) + 1,
        'n_clusters': len(sizes),
        'mean_cluster_size': np.mean(sizes),
        'n_switches': int(sum(sw)),
        'mean_irt': grp['irt_ms'].mean(),
        'mean_spam_dist': grp['spam_dist'].mean(),
        'mean_phon_sim': grp['phon_sim'].mean(),
    })
df_clusters = pd.DataFrame(cluster_rows)
df_clusters = df_clusters.merge(df_subjects[demog_cols], on='subject_id', how='left')
print(f'Cluster rows: {len(df_clusters)}')


Cluster rows: 99


In [15]:
# Save raw frames for inspection
df_subjects.to_csv(f'{DATAFRAMES_DIR}/subjects.csv', index=False)
df_trans.drop(columns=['cls_prev','cls_curr'], errors='ignore').to_csv(f'{DATAFRAMES_DIR}/transitions.csv', index=False)
df_words.to_csv(f'{DATAFRAMES_DIR}/words.csv', index=False)
df_clusters.to_csv(f'{DATAFRAMES_DIR}/clusters.csv', index=False)
print(f'Frames saved under {DATAFRAMES_DIR}/')


Frames saved under /content/phase2_outputs_complete/dataframes/


## Section 4 -- EDA

### EDA-A: Word inventory & script distribution

In [16]:
print('=' * 70)
print('EDA-A: WORD INVENTORY & SCRIPT')
print('=' * 70)

cls_by_domain = df_words.groupby(['domain','cls']).size().unstack(fill_value=0)
print('\nScript class counts per domain:')
print(cls_by_domain)

cls_by_domain_pct = cls_by_domain.div(cls_by_domain.sum(axis=1), axis=0) * 100
print('\nScript class proportions (%) per domain:')
print(cls_by_domain_pct.round(1))

# Top-20 words per domain
print('\nTop-20 words per domain (with class):')
for dom in sorted(df_words['domain'].unique()):
    sub = df_words[df_words['domain'] == dom]
    top = sub['word'].value_counts().head(20)
    print(f'\n  -- {dom} (n={len(sub)} word tokens, {sub["word"].nunique()} unique) --')
    for w, c in top.items():
        print(f'    {c:3d}  {w!r:20s}  [{classify_word(w, dom)}]')

# Plot script proportions
fig, ax = plt.subplots(figsize=(10, 5))
cls_by_domain_pct.plot(kind='bar', stacked=True, ax=ax,
                       color=PALETTE[:cls_by_domain_pct.shape[1]],
                       edgecolor='#333', width=0.7)
ax.set_ylabel('Proportion (%)')
ax.set_title('Script-class composition by domain')
ax.legend(title='class', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/EDA_A_script_composition.png', bbox_inches='tight')
plt.close()
print(f'\nSaved: EDA_A_script_composition.png')


EDA-A: WORD INVENTORY & SCRIPT

Script class counts per domain:
cls         hinglish  pure_devanagari  pure_english
domain                                             
animals           82              134           148
body-parts        36              119            49
colours           36                7           104
foods             95              123           105

Script class proportions (%) per domain:
cls         hinglish  pure_devanagari  pure_english
domain                                             
animals         22.5             36.8          40.7
body-parts      17.6             58.3          24.0
colours         24.5              4.8          70.7
foods           29.4             38.1          32.5

Top-20 words per domain (with class):

  -- animals (n=364 word tokens, 166 unique) --
     15  'कुत्ता'              [pure_devanagari]
     14  'शेर'                 [pure_devanagari]
     11  'हाथी'                [pure_devanagari]
     11  'बिल्ली'              [pur

### EDA-B: Demographics summary

In [17]:
print('=' * 70)
print('EDA-B: DEMOGRAPHICS')
print('=' * 70)

def vc(col, n=10):
    if col not in df_subjects.columns: return f'[col {col} missing]'
    return df_subjects[col].value_counts(dropna=False).head(n).to_string()

print(f'\nN subjects: {len(df_subjects)}')
print('\nAge distribution:')
if 'age' in df_subjects.columns:
    print(df_subjects['age'].describe().round(2).to_string())
print('\nGender (normalised):')
print(vc('gender_norm'))
print('\nFirst language:')
print(vc('first_language'))
print('\nL1 binary (Hindi vs Other):')
print(vc('l1_hindi'))
print('\nState/UT:')
print(vc('state_ut'))
print('\nEducation (years):')
if 'education' in df_subjects.columns:
    print(df_subjects['education'].describe().round(2).to_string())
print('\nDominant hand:')
print(vc('dominant_hand'))
print('\nAlert time / chronotype:')
print(vc('alert_time'))
print('\nLanguage count:')
print(vc('language_count'))
print('\nLikert means (1-5):')
for col in ['Hi_Read','Hi_Write','En_Read','En_Write','hi_fluency','en_fluency']:
    if col in df_subjects.columns:
        print(f'  {col:12s}: mean={df_subjects[col].mean():.2f}  sd={df_subjects[col].std():.2f}')

# Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_specs = [
    ('age', 'Age'), ('gender_norm', 'Gender'),
    ('first_language', 'First language'), ('alert_time', 'Chronotype'),
    ('dominant_hand', 'Dominant hand'), ('education', 'Education (yrs)')
]
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    if col not in df_subjects.columns:
        ax.text(0.5, 0.5, f'{col}\nmissing', transform=ax.transAxes, ha='center')
        ax.set_xticks([]); ax.set_yticks([])
        continue
    if df_subjects[col].dtype.kind in 'biufc':
        df_subjects[col].dropna().plot(kind='hist', ax=ax, bins=15, color=PALETTE[0], edgecolor='#333')
    else:
        df_subjects[col].fillna('NA').value_counts().plot(kind='bar', ax=ax,
                                                          color=PALETTE[0], edgecolor='#333')
        ax.tick_params(axis='x', rotation=30)
    ax.set_title(title)
plt.suptitle('Demographics (n=35; gender highly skewed)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/EDA_B_demographics.png', bbox_inches='tight')
plt.close()
print(f'\nSaved: EDA_B_demographics.png')


EDA-B: DEMOGRAPHICS

N subjects: 35

Age distribution:
count    35.00
mean     23.14
std       1.99
min      19.00
25%      21.50
50%      23.00
75%      24.50
max      27.00

Gender (normalised):
gender_norm
M    32
F     3

First language:
first_language
Hindi       15
English      5
Gujarati     5
hindi        4
Telugu       2
Punjabi      1
GUjarati     1
Hindi        1
Gujarti      1

L1 binary (Hindi vs Other):
l1_hindi
Hindi    20
Other    15

State/UT:
state_ut
Gujarat           7
Madhya Pradesh    6
Bihar             5
Maharashtra       4
Uttarakhand       2
Chhattisgarh      2
Rajasthan         2
Andhra Pradesh    1
Punjab            1
Delhi             1

Education (years):
count    35.00
mean     16.46
std       1.50
min      14.00
25%      16.00
50%      16.00
75%      17.00
max      20.00

Dominant hand:
dominant_hand
Right    31
Left      4

Alert time / chronotype:
alert_time
Morning          17
Evening          12
No difference     4
Afternoon         2

Language count

### EDA-C: Strategy text dump

In [20]:
print('=' * 70)
print('EDA-C: STRATEGIES')
print('=' * 70)

if 'strategies' not in df_subjects.columns:
    print('strategies field missing -- skipping')
else:
    strat = df_subjects[['subject_id','strategies']].dropna(subset=['strategies'])
    strat = strat[strat['strategies'].astype(str).str.strip() != '']
    print(f'\nFilled strategies: {len(strat)} / {len(df_subjects)}')
    for _, r in strat.iterrows():
        sid = str(r["subject_id"])
        print(f'\n  [{sid[:10]}] {r["strategies"]}')

    # Token frequency
    blob = ' '.join(strat['strategies'].astype(str).str.lower().tolist())
    tokens = re.findall(r"[a-zA-Zऀ-ॿ']+", blob)
    stop = set('the a an of to and or in on for is was were be by it its as i my'.split())
    tokens = [t for t in tokens if len(t) > 2 and t not in stop]
    counter = Counter(tokens)
    print('\n\nTop 30 strategy keywords:')
    for w, c in counter.most_common(30):
        print(f'  {c:3d}  {w}')


EDA-C: STRATEGIES

Filled strategies: 20 / 35

  [95712] it's based on my liking or based on the sequence increasing order of linking , power or shift in the colour from one shades to another and yes for the furniture i used the bedroom layout to order it 

  [92821] Memorization

  [73233] tried to think of domestic animals, then wild, etc. for body parts, went from top to bottom

  [83682] trying to recall comman items and trying to arrange them according how frequently i encounter them together or is one subset of another.

  [94502] I translated from english to hindi in google translate. 

  [81851] write word remembered correctly which we see in day to day life

  [83169] just recalling names based on my experience and real world

  [76112] I just kept typing the things that came to mind.

  [35389] clustering 

  [68981] wrote whatever came to top of my mind.

  [62335] Thinking of english/gujarati words and then thinking of its hindi

  [23853] I grouped the words based on their

### EDA-D: IRT distribution

In [21]:
print('=' * 70)
print('EDA-D: IRT DISTRIBUTION')
print('=' * 70)

# Pull all VFT IRTs (not just the kept transitions)
all_irts = []
for _, r in df_vft.iterrows():
    rts = r['irts']
    if isinstance(rts, list):
        all_irts.extend([x for x in rts if isinstance(x, (int, float))])
all_irts = np.array(all_irts)

print(f'Total IRT measurements: {len(all_irts)}')
print('Percentiles (ms):')
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f'  P{p:>2d}: {np.percentile(all_irts, p):>8.1f}')
print(f'  Min: {all_irts.min():.1f}   Max: {all_irts.max():.1f}')
print(f'  Filter (>0, <30000) keeps: {((all_irts>0)&(all_irts<30000)).mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(all_irts[(all_irts>0)&(all_irts<30000)], bins=60,
             color=PALETTE[0], edgecolor='#333')
axes[0].set_xlabel('IRT (ms)'); axes[0].set_ylabel('count'); axes[0].set_title('IRT histogram (filtered)')
axes[1].hist(np.log1p(all_irts[(all_irts>0)&(all_irts<30000)]), bins=60,
             color=PALETTE[1], edgecolor='#333')
axes[1].set_xlabel('log(1 + IRT)'); axes[1].set_ylabel('count'); axes[1].set_title('log-IRT histogram')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/EDA_D_irt_distribution.png', bbox_inches='tight')
plt.close()
print('\nSaved: EDA_D_irt_distribution.png')


EDA-D: IRT DISTRIBUTION
Total IRT measurements: 1044
Percentiles (ms):
  P 1:    917.2
  P 5:   1331.7
  P25:   2412.4
  P50:   4432.8
  P75:   7244.6
  P95:  14056.6
  P99:  24369.0
  Min: 732.8   Max: 42634.4
  Filter (>0, <30000) keeps: 99.43%

Saved: EDA_D_irt_distribution.png


### EDA-E: SpAM coordinate sanity

In [22]:
print('=' * 70)
print('EDA-E: SPAM SANITY')
print('=' * 70)
oob = 0; total = 0
for _, r in df_spam.iterrows():
    for wc in r['word_coords']:
        total += 1
        if not (0 <= wc['x'] <= 1 and 0 <= wc['y'] <= 1):
            oob += 1
print(f'Coordinates total: {total}; out-of-[0,1]: {oob} ({oob/total*100:.2f}%)')
print(f'\nWord-count per SpAM trial (describe):')
print(df_spam['word_count'].describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_spam['word_count'], bins=20, color=PALETTE[2], edgecolor='#333')
ax.set_xlabel('words placed in SpAM'); ax.set_ylabel('# trials')
ax.set_title('SpAM word-count distribution')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/EDA_E_spam_wordcount.png', bbox_inches='tight')
plt.close()
print('Saved: EDA_E_spam_wordcount.png')


EDA-E: SPAM SANITY
Coordinates total: 1033; out-of-[0,1]: 15 (1.45%)

Word-count per SpAM trial (describe):
count    105.00
mean       9.84
std        4.16
min        2.00
25%        7.00
50%        9.00
75%       12.00
max       24.00
Saved: EDA_E_spam_wordcount.png


## RQ1 -- Phonological similarity over retrieval position

In [23]:
print('=' * 70); print('RQ1'); print('=' * 70)

rq1_coef, rq1_pval = np.nan, np.nan
md_rq1 = None
df_trans['position_bin'] = pd.cut(df_trans['position'], bins=[0,4,8,12,50],
    labels=['Early (2-4)','Mid-Early (5-8)','Mid-Late (9-12)','Late (13+)'])
print(df_trans.groupby('position_bin', observed=True)['phon_sim'].agg(['mean','std','count']).round(4))

try:
    md_rq1 = smf.mixedlm('phon_sim ~ position_scaled', df_trans,
                         groups=df_trans['subject_id']).fit(reml=True)
    print(md_rq1.summary().tables[1])
    rq1_coef = md_rq1.params['position_scaled']
    rq1_pval = md_rq1.pvalues['position_scaled']
except Exception as e:
    print(f'RQ1 LMM failed: {e}')

within = df_trans[df_trans['is_switch'] == 0]['phon_sim']
switch = df_trans[df_trans['is_switch'] == 1]['phon_sim']
u_stat = u_pval = np.nan
if len(within) > 0 and len(switch) > 0:
    u_stat, u_pval = mannwhitneyu(within, switch, alternative='two-sided')
print(f'within M={within.mean():.4f}  switch M={switch.mean():.4f}  U={u_stat:.0f}  p={u_pval:.4g}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pos_means = df_trans.groupby('position')['phon_sim'].agg(['mean','sem']).reset_index()
pos_means = pos_means[pos_means['position'] <= 20]
axes[0].errorbar(pos_means['position'], pos_means['mean'], yerr=pos_means['sem'],
                 fmt='o-', color=PALETTE[0], capsize=3)
z = np.polyfit(pos_means['position'], pos_means['mean'], 1)
axes[0].plot(pos_means['position'], np.polyval(z, pos_means['position']),
             '--', color=PALETTE[1], label=f'beta={rq1_coef:.3f}, p={rq1_pval:.3f}')
axes[0].set_xlabel('Position'); axes[0].set_ylabel('Phon similarity'); axes[0].legend()
axes[0].set_title('A. Phon similarity vs position')

box_data = pd.DataFrame({'sim': list(within)+list(switch),
                         'type': ['Within']*len(within) + ['Switch']*len(switch)})
sns.boxplot(data=box_data, x='type', y='sim', hue='type',
            palette=[PALETTE[0], PALETTE[1]], ax=axes[1], legend=False)
axes[1].set_title(f'B. Within vs Switch (p={u_pval:.3g})'); axes[1].set_xlabel('')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ1_phon_over_position.png', bbox_inches='tight')
plt.close()
print('Saved: RQ1_phon_over_position.png')


RQ1
                   mean     std  count
position_bin                          
Early (2-4)      0.1931  0.1637    295
Mid-Early (5-8)  0.2026  0.1725    328
Mid-Late (9-12)  0.1510  0.1614    166
Late (13+)       0.1384  0.1521     99
                  Coef. Std.Err.       z  P>|z|  [0.025 0.975]
Intercept         0.200    0.014  14.668  0.000   0.173  0.226
position_scaled  -0.009    0.017  -0.549  0.583  -0.041  0.023
Group Var         0.003    0.006                              
within M=0.1869  switch M=0.1729  U=87319  p=0.3301
Saved: RQ1_phon_over_position.png


## RQ2 -- Joint cue model

In [24]:
print('=' * 70); print('RQ2'); print('=' * 70)
m1 = m2 = m3 = None

try:
    m1 = smf.mixedlm('log_irt ~ spam_dist_z', df_trans,
                     groups=df_trans['subject_id']).fit(reml=True)
    print(m1.summary().tables[1])
except Exception as e:
    print(f'M1 failed: {e}')

try:
    m2 = smf.mixedlm('log_irt ~ spam_dist_z + phon_sim + position_scaled', df_trans,
                     groups=df_trans['subject_id']).fit(reml=True)
    print(m2.summary().tables[1])
except Exception as e:
    print(f'M2 failed: {e}')

try:
    sub = df_trans.dropna(subset=['is_switch'])
    m3 = smf.mixedlm('log_irt ~ spam_dist_z + phon_sim * is_switch + position_scaled',
                     sub, groups=sub['subject_id']).fit(reml=True)
    print(m3.summary().tables[1])
except Exception as e:
    print(f'M3 failed: {e}')

def aic_bic(model):
    k = len(model.params)
    aic = 2*k - 2*model.llf
    bic = k*np.log(model.nobs) - 2*model.llf
    return aic, bic, k

comp_rows = []
for nm, mdl in [('M1: SpAM only', m1), ('M2: +Phon+Pos', m2), ('M3: +Switch interaction', m3)]:
    if mdl is None: continue
    a,b,k = aic_bic(mdl)
    comp_rows.append({'Model':nm,'AIC':round(a,1),'BIC':round(b,1),'LL':round(mdl.llf,1),'N':int(mdl.nobs)})
comp_df = pd.DataFrame(comp_rows)
print('\nModel comparison:'); print(comp_df.to_string(index=False))

# Robustness
robust = []
try:
    mr = smf.mixedlm('log_irt ~ spam_dist', df_trans, groups=df_trans['subject_id']).fit(reml=True)
    robust.append(('Raw distance', mr.params['spam_dist'], mr.pvalues['spam_dist']))
except Exception as e: pass
try:
    p95 = df_trans['irt_ms'].quantile(0.95)
    dw = df_trans.copy(); dw['log_irt_w'] = np.log1p(dw['irt_ms'].clip(upper=p95))
    mw = smf.mixedlm('log_irt_w ~ spam_dist_z', dw, groups=dw['subject_id']).fit(reml=True)
    robust.append(('Winsorized', mw.params['spam_dist_z'], mw.pvalues['spam_dist_z']))
except Exception as e: pass
try:
    df_no = df_trans[df_trans['position'] > 2]
    mn = smf.mixedlm('log_irt ~ spam_dist_z', df_no, groups=df_no['subject_id']).fit(reml=True)
    robust.append(('No first transition', mn.params['spam_dist_z'], mn.pvalues['spam_dist_z']))
except Exception as e: pass
print('\nRobustness:')
for name, b, p in robust:
    print(f'  {name:25s}  beta={b:.4f}  p={p:.4g}')

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
if m2 is not None:
    rows = []
    for p in ['spam_dist_z','phon_sim','position_scaled']:
        rows.append({'p': p, 'beta': m2.params[p], 'se': m2.bse[p], 'pv': m2.pvalues[p]})
    cdf = pd.DataFrame(rows)
    cols = [PALETTE[0] if p < 0.05 else '#CCCCCC' for p in cdf['pv']]
    ax.barh(cdf['p'], cdf['beta'], xerr=cdf['se']*1.96, color=cols, edgecolor='#333')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('M2: SpAM + Phon + Position coefficients (95% CI)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ2_joint_cue.png', bbox_inches='tight')
plt.close()
print('Saved: RQ2_joint_cue.png')


RQ2
             Coef. Std.Err.        z  P>|z| [0.025 0.975]
Intercept    8.457    0.077  109.980  0.000  8.307  8.608
spam_dist_z  0.097    0.018    5.430  0.000  0.062  0.132
Group Var    0.179    0.089                              
                  Coef. Std.Err.       z  P>|z|  [0.025 0.975]
Intercept         8.234    0.087  94.852  0.000   8.063  8.404
spam_dist_z       0.072    0.017   4.141  0.000   0.038  0.106
phon_sim         -0.148    0.115  -1.286  0.198  -0.372  0.077
position_scaled   0.486    0.056   8.663  0.000   0.376  0.596
Group Var         0.188    0.097                              
                     Coef. Std.Err.       z  P>|z|  [0.025 0.975]
Intercept            8.278    0.094  88.529  0.000   8.095  8.461
spam_dist_z          0.113    0.032   3.570  0.000   0.051  0.175
phon_sim            -0.106    0.133  -0.801  0.423  -0.367  0.154
is_switch           -0.086    0.084  -1.021  0.307  -0.252  0.079
phon_sim:is_switch  -0.159    0.243  -0.655  0.512  -0.6

## RQ3 -- Domain differences

In [25]:
print('=' * 70); print('RQ3'); print('=' * 70)
domain_counts = df_clusters['domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 10].index.tolist()
df_clust_valid = df_clusters[df_clusters['domain'].isin(valid_domains)]
print(f'Valid domains (>=10 trials): {valid_domains}')
print(df_clust_valid.groupby('domain').agg({
    'n_words':['mean','std'], 'mean_cluster_size':['mean','std'],
    'n_switches':['mean','std'], 'mean_irt':['mean','std'],
    'mean_phon_sim':['mean','std'], 'mean_spam_dist':['mean','std']
}).round(3))

kw_results = []
for var in ['mean_cluster_size','n_words','mean_irt','mean_phon_sim','n_switches']:
    groups = [g[var].dropna().values for _, g in df_clust_valid.groupby('domain')]
    if len(groups) >= 2 and all(len(g) > 0 for g in groups):
        h, p = kruskal(*groups)
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        kw_results.append({'Variable':var,'H':h,'p':p,'Significant':sig})
        print(f'  {var:20s} H={h:.2f}  p={p:.4g}  {sig}')
kw_df = pd.DataFrame(kw_results)

# Robustness with mean-split
print('\nMean-split robustness:')
for var in ['is_switch_mean']:
    pass  # use is_switch_mean to rebuild
cluster_rows_mean = []
for (subj, dom), grp in df_trans.groupby(['subject_id','domain']):
    sw = grp['is_switch_mean'].dropna().values
    if len(sw) == 0: continue
    sizes = []; cur = 1
    for s in sw:
        if s == 0: cur += 1
        else: sizes.append(cur); cur = 1
    sizes.append(cur)
    cluster_rows_mean.append({'subject_id':subj,'domain':dom,
        'mean_cluster_size':np.mean(sizes),'n_switches':int(sum(sw))})
df_clust_mean = pd.DataFrame(cluster_rows_mean)
df_clust_mean = df_clust_mean[df_clust_mean['domain'].isin(valid_domains)]
for var in ['mean_cluster_size','n_switches']:
    groups = [g[var].dropna().values for _, g in df_clust_mean.groupby('domain')]
    if len(groups) >= 2:
        h, p = kruskal(*groups)
        print(f'  {var:20s} (mean-split) H={h:.2f}  p={p:.4g}')

# Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
plot_vars = [('n_words','Total words'),('mean_cluster_size','Cluster size'),
             ('n_switches','# switches'),('mean_irt','Mean IRT (ms)'),
             ('mean_phon_sim','Mean phon sim'),('mean_spam_dist','Mean SpAM dist')]
for ax, (v, t) in zip(axes.ravel(), plot_vars):
    sns.boxplot(data=df_clust_valid, x='domain', y=v, hue='domain',
                palette=PALETTE[:len(valid_domains)], ax=ax, legend=False)
    sns.stripplot(data=df_clust_valid, x='domain', y=v, color='black',
                  alpha=0.3, size=3, ax=ax, jitter=True)
    ax.set_title(t); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=15)
    row = kw_df[kw_df['Variable'] == v]
    if not row.empty:
        ax.text(0.98, 0.98, f"KW p={row.iloc[0]['p']:.3g}", transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ3_domain_differences.png', bbox_inches='tight')
plt.close()
print('Saved: RQ3_domain_differences.png')


RQ3
Valid domains (>=10 trials): ['animals', 'foods', 'body-parts', 'colours']
           n_words        mean_cluster_size        n_switches         \
              mean    std              mean    std       mean    std   
domain                                                                 
animals     10.394  4.756             3.228  1.709      2.636  1.729   
body-parts   8.591  3.217             3.198  1.880      2.227  1.824   
colours     13.364  3.414             2.293  0.590      5.091  2.212   
foods        9.333  4.136             3.084  1.601      2.455  1.523   

            mean_irt           mean_phon_sim        mean_spam_dist         
                mean       std          mean    std           mean    std  
domain                                                                     
animals     5770.975  2257.267         0.170  0.060          0.249  0.135  
body-parts  6270.837  2107.890         0.265  0.088          0.187  0.101  
colours     4042.920  1598.311      

## RQ4 -- Fluency x Domain x Position GLM (with decomposed Likerts)

In [26]:
print('=' * 70); print('RQ4'); print('=' * 70)
df_glm = df_trans[df_trans['domain'].isin(valid_domains)].dropna(
    subset=['hi_fluency','log_irt','position_scaled']).copy()
df_glm['hi_fluency_c'] = df_glm['hi_fluency'] - df_glm['hi_fluency'].mean()
df_glm['domain'] = pd.Categorical(df_glm['domain'])
print(f'GLM N: {len(df_glm)} obs across {df_glm["subject_id"].nunique()} subjects')

glm_a = glm_b = glm_decomp = None
try:
    glm_a = smf.mixedlm('log_irt ~ hi_fluency_c + C(domain) + position_scaled',
                        df_glm, groups=df_glm['subject_id']).fit(reml=True)
    print(glm_a.summary().tables[1])
except Exception as e: print(f'GLM A failed: {e}')

try:
    glm_b = smf.mixedlm(
        'log_irt ~ hi_fluency_c * position_scaled + hi_fluency_c * C(domain) + C(domain) * position_scaled',
        df_glm, groups=df_glm['subject_id']).fit(reml=True)
    print(glm_b.summary().tables[1])
except Exception as e: print(f'GLM B failed: {e}')

# Decompose: Hi_Read & Hi_Write separately
df_glm['Hi_Read_c']  = df_glm['Hi_Read']  - df_glm['Hi_Read'].mean()  if 'Hi_Read'  in df_glm.columns else 0
df_glm['Hi_Write_c'] = df_glm['Hi_Write'] - df_glm['Hi_Write'].mean() if 'Hi_Write' in df_glm.columns else 0
try:
    glm_decomp = smf.mixedlm(
        'log_irt ~ Hi_Read_c * C(domain) + Hi_Write_c * C(domain) + position_scaled',
        df_glm.dropna(subset=['Hi_Read','Hi_Write']),
        groups=df_glm.dropna(subset=['Hi_Read','Hi_Write'])['subject_id']).fit(reml=True)
    print('\nDecomposed Likerts:')
    print(glm_decomp.summary().tables[1])
except Exception as e: print(f'GLM decomp failed: {e}')

if glm_a is not None and glm_b is not None:
    aA = aic_bic(glm_a); aB = aic_bic(glm_b)
    print(f'\nAIC A={aA[0]:.1f}  B={aB[0]:.1f}  preferred={"B" if aB[0]<aA[0] else "A"}')

# Forest plot of best model
fig, ax = plt.subplots(figsize=(10, 7))
best = glm_b if (glm_b is not None and glm_a is not None and aic_bic(glm_b)[0] < aic_bic(glm_a)[0]) else (glm_a or glm_decomp)
if best is not None:
    p = best.params.drop(['Intercept','Group Var'], errors='ignore')
    se = best.bse.reindex(p.index); pv = best.pvalues.reindex(p.index)
    names = [n.replace('C(domain)[T.','').replace(']','') for n in p.index]
    cols = [PALETTE[0] if x<0.05 else '#BBBBBB' for x in pv]
    ax.barh(range(len(p)), p.values, xerr=se*1.96, color=cols, edgecolor='#333')
    ax.set_yticks(range(len(p))); ax.set_yticklabels(names, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('beta'); ax.set_title('RQ4 best-model coefficients (95% CI)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ4_glm.png', bbox_inches='tight')
plt.close()
print('Saved: RQ4_glm.png')


RQ4
GLM N: 888 obs across 33 subjects
                          Coef. Std.Err.       z  P>|z|  [0.025 0.975]
Intercept                 8.153    0.086  95.252  0.000   7.985  8.320
C(domain)[T.body-parts]  -0.050    0.056  -0.893  0.372  -0.160  0.060
C(domain)[T.colours]      0.007    0.060   0.118  0.906  -0.111  0.126
C(domain)[T.foods]        0.004    0.045   0.081  0.935  -0.085  0.092
hi_fluency_c              0.177    0.104   1.702  0.089  -0.027  0.380
position_scaled           0.525    0.056   9.375  0.000   0.415  0.635
Group Var                 0.181    0.094                              
                                          Coef. Std.Err.       z  P>|z|  \
Intercept                                 8.151    0.093  87.381  0.000   
C(domain)[T.body-parts]                  -0.078    0.096  -0.814  0.415   
C(domain)[T.colours]                     -0.168    0.106  -1.584  0.113   
C(domain)[T.foods]                        0.102    0.082   1.241  0.215   
hi_fluency_c       

## RQ5 -- Script choice & Hinglish penalty

In [27]:
print('=' * 70); print('RQ5: SCRIPT CHOICE & HINGLISH'); print('=' * 70)

# Per (subject, domain) script proportions
def _frac(s, lvl):
    return (s == lvl).mean()

script_per_trial = df_words.groupby(['subject_id','domain']).agg(
    n_words=('word','count'),
    pct_devanagari=('cls', lambda s: _frac(s, 'pure_devanagari')*100),
    pct_english=('cls', lambda s: _frac(s, 'pure_english')*100),
    pct_hinglish=('cls', lambda s: _frac(s, 'hinglish')*100),
).reset_index()
script_per_trial = script_per_trial.merge(df_subjects[demog_cols], on='subject_id', how='left')
print('\nMean script % per domain:')
print(script_per_trial.groupby('domain')[['pct_devanagari','pct_english','pct_hinglish']].mean().round(1))

# Predict pct_devanagari from L1 + Likerts + domain (mixed-effects)
print('\n--- Predicting pct_devanagari ---')
sub = script_per_trial.dropna(subset=['pct_devanagari','l1_hindi','Hi_Read','Hi_Write','En_Read','En_Write']).copy()
m_dev = None
if len(sub) > 0:
    try:
        m_dev = smf.mixedlm('pct_devanagari ~ C(l1_hindi) + Hi_Read + Hi_Write + En_Read + En_Write + C(domain)',
                             sub, groups=sub['subject_id']).fit(reml=True)
        print(m_dev.summary().tables[1])
    except Exception as e:
        print(f'm_dev failed: {e}')

# IRT by transition class -- compare {pure_devanagari, pure_english, hinglish} prev->curr same-class
print('\n--- IRT by transition class (same-class pairs only) ---')
df_trans['trans_class'] = np.where(df_trans['cls_prev'] == df_trans['cls_curr'],
                                    df_trans['cls_curr'], 'mixed')
print(df_trans.groupby('trans_class')['irt_ms'].agg(['mean','median','count']).round(1))
for c1, c2 in combinations(['pure_devanagari','pure_english','hinglish','mixed'], 2):
    g1 = df_trans[df_trans['trans_class']==c1]['irt_ms']
    g2 = df_trans[df_trans['trans_class']==c2]['irt_ms']
    if len(g1) > 5 and len(g2) > 5:
        u, p = mannwhitneyu(g1, g2, alternative='two-sided')
        print(f'  {c1:18s} vs {c2:18s}  U={u:.0f}  p={p:.4g}')

# Mixed-script transition cost
script_changed = df_trans[df_trans['trans_class'] == 'mixed']
print(f'\nMixed-script transitions: n={len(script_changed)}')
if len(script_changed) > 5:
    g_mix  = script_changed['irt_ms']
    g_same = df_trans[df_trans['trans_class'] != 'mixed']['irt_ms']
    u, p = mannwhitneyu(g_mix, g_same, alternative='two-sided')
    print(f'  same-class M={g_same.mean():.0f}  mixed M={g_mix.mean():.0f}  U={u:.0f}  p={p:.4g}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=script_per_trial.melt(id_vars=['domain'],
            value_vars=['pct_devanagari','pct_english','pct_hinglish']),
            x='domain', y='value', hue='variable',
            palette=PALETTE[:3], ax=axes[0])
axes[0].set_ylabel('% words'); axes[0].set_title('A. Script class % by domain (per-trial)')
axes[0].legend(title='', loc='upper right')

sns.boxplot(data=df_trans, x='trans_class', y='log_irt',
            order=['pure_devanagari','pure_english','hinglish','mixed'],
            hue='trans_class', palette=PALETTE[:4], ax=axes[1], legend=False)
axes[1].set_title('B. log-IRT by transition class')
axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ5_script_hinglish.png', bbox_inches='tight')
plt.close()
print('Saved: RQ5_script_hinglish.png')


RQ5: SCRIPT CHOICE & HINGLISH

Mean script % per domain:
            pct_devanagari  pct_english  pct_hinglish
domain                                               
animals               51.4         28.2          20.4
body-parts            70.8         17.1          12.1
colours                9.1         64.4          26.5
foods                 51.0         25.4          23.6

--- Predicting pct_devanagari ---
                            Coef. Std.Err.       z  P>|z|    [0.025  0.975]
Intercept                 -80.611   58.809  -1.371  0.170  -195.874  34.652
C(l1_hindi)[T.Other]        5.106   17.881   0.286  0.775   -29.941  40.152
C(domain)[T.body-parts]     0.100    0.389   0.257  0.797    -0.662   0.861
C(domain)[T.colours]       -0.218    0.544  -0.401  0.689    -1.285   0.849
C(domain)[T.foods]         -0.408    0.335  -1.218  0.223    -1.065   0.249
Hi_Read                    16.300   12.426   1.312  0.190    -8.055  40.654
Hi_Write                   10.184   10.465   0.973  

## RQ6 -- Embedding-based semantic distance

In [28]:
print('=' * 70); print('RQ6: EMBEDDING-BASED SEMANTIC DISTANCE'); print('=' * 70)

# Collect ALL unique words actually appearing in transitions
all_words = set(df_trans['word_prev']).union(df_trans['word_curr'])
all_words = [w for w in all_words if isinstance(w, str) and w.strip()]
print(f'Unique words to embed: {len(all_words)}')

emb_cache = load_or_compute_embeddings(all_words, cache_path=EMB_CACHE_PATH)
MODEL_KEYS = list(emb_cache.keys())
print(f'\nModels in cache: {MODEL_KEYS}')

# Compute per-transition cosine distance for each model
for key in MODEL_KEYS:
    short = key.split('/')[-1].replace('paraphrase-multilingual-','').replace('-base-cased','')
    short = short.replace('-', '_')
    col = f'embed_{short}'
    embs = emb_cache[key]
    df_trans[col] = df_trans.apply(
        lambda r: cosine_dist(embs.get(r['word_prev']), embs.get(r['word_curr'])), axis=1)
    print(f'  {col}: mean={df_trans[col].mean():.3f} sd={df_trans[col].std():.3f} n_valid={df_trans[col].notna().sum()}')

emb_cols = [c for c in df_trans.columns if c.startswith('embed_')]
print(f'\nEmbedding columns: {emb_cols}')

# z-score each embedding distance for fair comparison
for c in emb_cols:
    df_trans[c + '_z'] = (df_trans[c] - df_trans[c].mean()) / df_trans[c].std()

# Correlation matrix among distance metrics
dist_cols = ['spam_dist','phon_sim'] + emb_cols
corr_mat = df_trans[dist_cols].corr(method='spearman')
print('\nSpearman correlation among distance metrics:')
print(corr_mat.round(3))

# Headline test: per model, fit log_irt ~ spam + phon + position + embed
print('\n--- Per-model joint fit ---')
embed_models = {}
for c in emb_cols:
    cz = c + '_z'
    sub = df_trans.dropna(subset=['log_irt','spam_dist_z','phon_sim','position_scaled', cz])
    try:
        m = smf.mixedlm(f'log_irt ~ spam_dist_z + phon_sim + position_scaled + {cz}',
                        sub, groups=sub['subject_id']).fit(reml=True)
        embed_models[c] = m
        a, b, k = aic_bic(m)
        print(f'\n{c}:')
        print(f'  AIC={a:.1f}  beta_embed={m.params[cz]:.4f}  p={m.pvalues[cz]:.4g}')
        print(f'  beta_spam={m.params["spam_dist_z"]:.4f}  p={m.pvalues["spam_dist_z"]:.4g}')
    except Exception as e:
        print(f'  {c}: fit failed: {e}')

# All-four model
m_all = None
all_z_cols = [c + '_z' for c in emb_cols]
if len(all_z_cols) >= 1:
    try:
        sub = df_trans.dropna(subset=['log_irt','spam_dist_z','phon_sim','position_scaled'] + all_z_cols)
        formula = 'log_irt ~ spam_dist_z + phon_sim + position_scaled + ' + ' + '.join(all_z_cols)
        m_all = smf.mixedlm(formula, sub, groups=sub['subject_id']).fit(reml=True)
        print('\n--- All-models joint fit ---')
        print(m_all.summary().tables[1])
        a, b, k = aic_bic(m_all)
        print(f'AIC={a:.1f}  BIC={b:.1f}  N={int(m_all.nobs)}')
    except Exception as e:
        print(f'All-models fit failed: {e}')

# Comparison vs baseline (just SpAM + Phon + Pos = M2)
if m2 is not None:
    print(f'\nBaseline M2 AIC = {aic_bic(m2)[0]:.1f}')
    if m_all is not None:
        print(f'All-embeddings AIC = {aic_bic(m_all)[0]:.1f}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('A. Spearman correlations among distance metrics')

if m_all is not None:
    p = m_all.params.drop(['Intercept','Group Var'], errors='ignore')
    se = m_all.bse.reindex(p.index); pv = m_all.pvalues.reindex(p.index)
    names = list(p.index)
    cols = [PALETTE[0] if x < 0.05 else '#BBBBBB' for x in pv]
    axes[1].barh(range(len(p)), p.values, xerr=se*1.96, color=cols, edgecolor='#333')
    axes[1].set_yticks(range(len(p))); axes[1].set_yticklabels(names, fontsize=8)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title('B. All-distance joint model coefficients')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ6_embeddings.png', bbox_inches='tight')
plt.close()
print('Saved: RQ6_embeddings.png')


RQ6: EMBEDDING-BASED SEMANTIC DISTANCE
Unique words to embed: 533
  Embedding device: cpu
\n  Computing 533 embeddings with paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Models in cache: ['paraphrase-multilingual-MiniLM-L12-v2', 'sentence-transformers/LaBSE', 'google/muril-base-cased']
  embed_MiniLM_L12_v2: mean=0.499 sd=0.169 n_valid=888
  embed_LaBSE: mean=0.447 sd=0.157 n_valid=888
  embed_muril: mean=0.007 sd=0.004 n_valid=888

Embedding columns: ['embed_MiniLM_L12_v2', 'embed_LaBSE', 'embed_muril']

Spearman correlation among distance metrics:
                     spam_dist  phon_sim  embed_MiniLM_L12_v2  embed_LaBSE  \
spam_dist                1.000    -0.103                0.213        0.061   
phon_sim                -0.103     1.000               -0.128       -0.158   
embed_MiniLM_L12_v2      0.213    -0.128                1.000       -0.004   
embed_LaBSE              0.061    -0.158               -0.004        1.000   
embed_muril              0.029    -0.197               -0.223        0.602   

                     embed_muril  
spam_dist                  0.029  
phon_sim                  -0.197  
embed_MiniLM_L12_v2       -0.223  
embe

## RQ7 -- Patch foraging (MVT)

In [29]:
print('=' * 70); print('RQ7: MVT'); print('=' * 70)

# 3-way agreement
sub = df_trans.dropna(subset=['is_switch','is_switch_mean','is_switch_mvt']).copy()
sub['is_switch']      = sub['is_switch'].astype(int)
sub['is_switch_mean'] = sub['is_switch_mean'].astype(int)
sub['is_switch_mvt']  = sub['is_switch_mvt'].astype(int)

print(f'N transitions with all 3 indicators: {len(sub)}')
chi2 = np.nan; p_chi = np.nan
ctab = None
if len(sub) == 0:
    print('No transitions with all switch indicators -- skipping rest of RQ7')
else:
    print('\nMVT switch rate per domain:')
    print(sub.groupby('domain')['is_switch_mvt'].mean().round(3))

    print('\nAgreement rates (% of transitions classified the same):')
    print(f'  median <-> mean : {(sub["is_switch"] == sub["is_switch_mean"]).mean()*100:.1f}%')
    print(f'  median <-> mvt  : {(sub["is_switch"] == sub["is_switch_mvt"]).mean()*100:.1f}%')
    print(f'  mean   <-> mvt  : {(sub["is_switch_mean"] == sub["is_switch_mvt"]).mean()*100:.1f}%')

    # Confusion (median vs mvt)
    ctab = pd.crosstab(sub['is_switch'], sub['is_switch_mvt'],
                       rownames=['SpAM-median'], colnames=['MVT'])
    print('\nConfusion (median vs MVT):'); print(ctab)
    try:
        if ctab.shape == (2, 2) and ctab.values.sum() > 0:
            chi2, p_chi, _, _ = chi2_contingency(ctab)
            print(f'chi2={chi2:.2f}  p={p_chi:.4g}')
    except Exception as e:
        print(f'chi2 failed: {e}')

# IRT spike at switch+1 vs switch-1 for MVT
print('\nMVT IRT spike test (paired Wilcoxon over per-trial means):')
spike_pairs = []
for (subj, dom), grp in df_trans.groupby(['subject_id','domain']):
    g = grp.sort_values('position').reset_index(drop=True)
    for i, row in g.iterrows():
        if row['is_switch_mvt'] == 1 and 0 < i < len(g) - 1:
            spike_pairs.append((g.iloc[i-1]['irt_ms'], g.iloc[i+1]['irt_ms']))
if spike_pairs:
    pre  = np.array([p[0] for p in spike_pairs])
    post = np.array([p[1] for p in spike_pairs])
    try:
        wstat, wp = wilcoxon(post, pre, alternative='greater')
        print(f'  pre M={pre.mean():.0f}  post M={post.mean():.0f}  W={wstat:.0f}  p={wp:.4g}')
    except Exception as e:
        print(f'  Wilcoxon failed: {e}')

# Cluster sizes under MVT
mvt_clusters = []
for (subj, dom), grp in df_trans.groupby(['subject_id','domain']):
    sw = grp['is_switch_mvt'].dropna().values
    if len(sw) == 0: continue
    sizes = []; cur = 1
    for s in sw:
        if s == 0: cur += 1
        else: sizes.append(cur); cur = 1
    sizes.append(cur)
    mvt_clusters.append({'subject_id':subj,'domain':dom,
        'mean_cluster_mvt':np.mean(sizes), 'n_switches_mvt':int(sum(sw))})
df_mvt = pd.DataFrame(mvt_clusters)
print('\nCluster size by domain (MVT):')
print(df_mvt.groupby('domain')['mean_cluster_mvt'].describe().round(2))

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(ctab, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('A. SpAM-median vs MVT switch confusion')

dft = df_mvt.merge(df_clusters[['subject_id','domain','mean_cluster_size','n_switches']],
                    on=['subject_id','domain'], how='inner')
axes[1].scatter(dft['mean_cluster_size'], dft['mean_cluster_mvt'],
                c=[PALETTE[0]], edgecolors='#333', s=40, alpha=0.7)
lim = max(dft['mean_cluster_size'].max(), dft['mean_cluster_mvt'].max()) + 0.5
axes[1].plot([0, lim], [0, lim], '--', color='grey', alpha=0.5)
axes[1].set_xlabel('Cluster size (SpAM-median)')
axes[1].set_ylabel('Cluster size (MVT)')
axes[1].set_title('B. Per-trial cluster size: SpAM vs MVT')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ7_mvt.png', bbox_inches='tight')
plt.close()
print('Saved: RQ7_mvt.png')


RQ7: MVT
N transitions with all 3 indicators: 888

MVT switch rate per domain:
domain
animals       0.371
body-parts    0.365
colours       0.353
foods         0.218
Name: is_switch_mvt, dtype: float64

Agreement rates (% of transitions classified the same):
  median <-> mean : 96.5%
  median <-> mvt  : 60.2%
  mean   <-> mvt  : 60.1%

Confusion (median vs MVT):
MVT            0    1
SpAM-median          
0            433  182
1            171  102
chi2=4.90  p=0.02693

MVT IRT spike test (paired Wilcoxon over per-trial means):
  pre M=4145  post M=5390  W=17640  p=4.171e-06

Cluster size by domain (MVT):
            count  mean   std   min   25%   50%   75%    max
domain                                                      
animals      33.0  2.97  1.88  1.29  1.89  2.25  3.33  10.00
body-parts   22.0  3.05  1.76  1.20  1.79  2.83  3.80   8.00
colours      11.0  2.81  1.14  1.56  1.90  2.33  3.42   4.75
foods        33.0  4.26  2.42  1.20  2.33  4.00  5.33  11.00
Saved: RQ7_mvt.png


## RQ8 -- Word typicality / production frequency

In [30]:
print('=' * 70); print('RQ8: WORD TYPICALITY'); print('=' * 70)

# Build per-domain word frequency = # subjects who produced this word (any time)
prod_freq = {}  # (domain, word) -> count
n_subj_per_domain = df_words.groupby('domain')['subject_id'].nunique().to_dict()
for dom, grp in df_words.groupby('domain'):
    sub_words = grp.groupby('word')['subject_id'].nunique()
    for w, c in sub_words.items():
        prod_freq[(dom, w)] = c

df_words['prod_freq'] = df_words.apply(
    lambda r: prod_freq.get((r['domain'], r['word']), 0), axis=1)
df_words['prod_freq_pct'] = df_words.apply(
    lambda r: prod_freq.get((r['domain'], r['word']), 0) / n_subj_per_domain.get(r['domain'], 1) * 100,
    axis=1)
df_words['prod_freq_log'] = np.log1p(df_words['prod_freq'])

# Also assign to df_trans for the prev-word and curr-word
df_trans['curr_freq'] = df_trans.apply(
    lambda r: prod_freq.get((r['domain'], r['word_curr']), 0), axis=1)
df_trans['curr_freq_log'] = np.log1p(df_trans['curr_freq'])

print('\nProduction-frequency descriptives (curr word per transition):')
print(df_trans.groupby('domain')['curr_freq'].describe().round(1))

# Mixed-effects: log_irt ~ curr_freq_log + spam_dist_z + position
m_freq = None
try:
    sub = df_trans.dropna(subset=['log_irt','curr_freq_log','spam_dist_z','position_scaled'])
    m_freq = smf.mixedlm('log_irt ~ curr_freq_log + spam_dist_z + position_scaled',
                         sub, groups=sub['subject_id']).fit(reml=True)
    print('\nlog_irt ~ curr_freq_log + spam_dist_z + position_scaled:')
    print(m_freq.summary().tables[1])
except Exception as e:
    print(f'm_freq failed: {e}')

# Body-parts story: do high-fluency speakers retrieve rarer items?
print('\n--- Body-parts: typicality x fluency ---')
bp = df_words[df_words['domain'] == 'body-parts'].dropna(subset=['hi_fluency'])
if len(bp) > 0:
    bp_per_trial = bp.groupby('subject_id').agg(
        mean_typicality=('prod_freq_pct','mean'),
        hi_fluency=('hi_fluency','first')).reset_index()
    if len(bp_per_trial) > 3:
        r, p = pearsonr(bp_per_trial['hi_fluency'], bp_per_trial['mean_typicality'])
        print(f'  body-parts: corr(hi_fluency, mean_typicality) r={r:.3f}  p={p:.4g}  n={len(bp_per_trial)}')

# Same for all domains
print('\nTypicality x fluency per domain (Pearson):')
for dom in sorted(df_words['domain'].unique()):
    sub = df_words[df_words['domain'] == dom].dropna(subset=['hi_fluency'])
    if len(sub) == 0: continue
    per_subj = sub.groupby('subject_id').agg(
        mean_typicality=('prod_freq_pct','mean'),
        hi_fluency=('hi_fluency','first')).reset_index()
    if len(per_subj) > 3:
        r, p = pearsonr(per_subj['hi_fluency'], per_subj['mean_typicality'])
        print(f'  {dom:12s}: r={r:.3f}  p={p:.4g}  n={len(per_subj)}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df_words, x='prod_freq_pct', hue='domain', ax=axes[0],
             palette=PALETTE[:df_words['domain'].nunique()],
             multiple='dodge', bins=20)
axes[0].set_xlabel('Production frequency (% subjects)')
axes[0].set_title('A. Typicality distribution by domain')

dom_subj = df_words.dropna(subset=['hi_fluency']).groupby(['subject_id','domain']).agg(
    mean_typ=('prod_freq_pct','mean'),
    hi_fluency=('hi_fluency','first')).reset_index()
sns.scatterplot(data=dom_subj, x='hi_fluency', y='mean_typ', hue='domain',
                palette=PALETTE[:dom_subj['domain'].nunique()], ax=axes[1])
axes[1].set_xlabel('Hindi fluency (avg Likert)'); axes[1].set_ylabel('Mean typicality (% subjects)')
axes[1].set_title('B. Mean typicality vs hi_fluency (per subject x domain)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ8_typicality.png', bbox_inches='tight')
plt.close()
print('Saved: RQ8_typicality.png')


RQ8: WORD TYPICALITY

Production-frequency descriptives (curr word per transition):
            count  mean  std  min  25%  50%  75%   max
domain                                                
animals     310.0   4.0  3.7  1.0  1.0  2.0  5.0  15.0
body-parts  167.0   4.8  4.6  1.0  1.0  3.0  8.0  14.0
colours     136.0   3.0  2.0  1.0  1.0  2.0  5.0   6.0
foods       275.0   2.7  2.9  1.0  1.0  1.0  3.0  14.0

log_irt ~ curr_freq_log + spam_dist_z + position_scaled:
                  Coef. Std.Err.       z  P>|z|  [0.025  0.975]
Intercept         8.408    0.106  79.244  0.000   8.200   8.616
curr_freq_log    -0.124    0.036  -3.496  0.000  -0.194  -0.055
spam_dist_z       0.074    0.017   4.318  0.000   0.041   0.108
position_scaled   0.432    0.058   7.466  0.000   0.318   0.545
Group Var         0.216    0.113                               

--- Body-parts: typicality x fluency ---
  body-parts: corr(hi_fluency, mean_typicality) r=0.602  p=0.001872  n=24

Typicality x fluency per do

## RQ9 -- Individual-difference battery

In [31]:
print('=' * 70); print('RQ9: INDIVIDUAL DIFFERENCES'); print('=' * 70)

# Per-trial summary frame
trial_sum = df_clusters[['subject_id','domain','n_words','mean_irt']].merge(
    df_subjects, on='subject_id', how='left')
trial_sum = trial_sum.merge(
    script_per_trial[['subject_id','domain','pct_devanagari','pct_english','pct_hinglish']],
    on=['subject_id','domain'], how='left')
print(f'Trial-level rows: {len(trial_sum)}')

# 1. L1 effects
print('\n--- L1 (Hindi vs Other) effects ---')
for var in ['n_words','mean_irt','pct_devanagari']:
    sub = trial_sum.dropna(subset=[var,'l1_hindi'])
    if sub['l1_hindi'].nunique() < 2: continue
    g_hi = sub[sub['l1_hindi']=='Hindi'][var].values
    g_oth = sub[sub['l1_hindi']=='Other'][var].values
    if len(g_hi) > 3 and len(g_oth) > 3:
        u, p = mannwhitneyu(g_hi, g_oth, alternative='two-sided')
        print(f'  {var:18s} Hindi M={g_hi.mean():.2f}  Other M={g_oth.mean():.2f}  U={u:.0f}  p={p:.4g}')

# 2. Decoupled Likerts -- mixed-effects: word_count ~ Hi_Read + Hi_Write + En_Read + En_Write
print('\n--- Decoupled Likerts -> n_words ---')
sub = trial_sum.dropna(subset=['n_words','Hi_Read','Hi_Write','En_Read','En_Write'])
m_likert = None
if len(sub) > 10:
    try:
        m_likert = smf.mixedlm('n_words ~ Hi_Read + Hi_Write + En_Read + En_Write + C(domain)',
                                sub, groups=sub['subject_id']).fit(reml=True)
        print(m_likert.summary().tables[1])
    except Exception as e:
        print(f'  failed: {e}')

# 3. Chronotype x position
print('\n--- Chronotype x position interaction (log_irt) ---')
sub = df_trans.dropna(subset=['log_irt','position_scaled','alert_time'])
if sub['alert_time'].nunique() >= 2:
    try:
        m_chrono = smf.mixedlm('log_irt ~ C(alert_time) * position_scaled',
                                sub, groups=sub['subject_id']).fit(reml=True)
        print(m_chrono.summary().tables[1])
    except Exception as e:
        print(f'  failed: {e}')

# 4. Education
print('\n--- Education years effects ---')
for var in ['n_words','mean_irt']:
    sub = trial_sum.dropna(subset=[var,'education'])
    if len(sub) > 5:
        r, p = spearmanr(sub['education'], sub[var])
        print(f'  {var:18s}: rho={r:.3f}  p={p:.4g}  n={len(sub)}')

# 5. Gender (caveat)
print('\n--- Gender effects (CAVEAT n_F=3) ---')
for var in ['n_words','mean_irt']:
    sub = trial_sum.dropna(subset=[var,'gender_norm'])
    if sub['gender_norm'].nunique() < 2: continue
    g_m = sub[sub['gender_norm']=='M'][var].values
    g_f = sub[sub['gender_norm']=='F'][var].values
    if len(g_m) > 3 and len(g_f) > 0:
        u, p = mannwhitneyu(g_m, g_f, alternative='two-sided')
        print(f'  {var:18s} M (n={len(g_m)}) M={g_m.mean():.2f}  F (n={len(g_f)}) M={g_f.mean():.2f}  U={u:.0f}  p={p:.4g}')

# Plot summary: L1 split
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, var in zip(axes, ['n_words','mean_irt','pct_devanagari']):
    sub = trial_sum.dropna(subset=[var,'l1_hindi'])
    if sub['l1_hindi'].nunique() < 2:
        ax.text(0.5, 0.5, 'L1 missing', transform=ax.transAxes, ha='center'); continue
    sns.boxplot(data=sub, x='l1_hindi', y=var, hue='l1_hindi',
                palette=[PALETTE[0], PALETTE[3]], ax=ax, legend=False)
    sns.stripplot(data=sub, x='l1_hindi', y=var, color='black', alpha=0.4, size=3, ax=ax)
    ax.set_title(f'{var} by L1')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ9_individual_diffs.png', bbox_inches='tight')
plt.close()
print('Saved: RQ9_individual_diffs.png')


RQ9: INDIVIDUAL DIFFERENCES
Trial-level rows: 99

--- L1 (Hindi vs Other) effects ---
  n_words            Hindi M=9.63  Other M=10.43  U=988  p=0.1377
  mean_irt           Hindi M=5916.06  Other M=5435.98  U=1260  p=0.6581
  pct_devanagari     Hindi M=47.37  Other M=49.66  U=1179  p=0.8867

--- Decoupled Likerts -> n_words ---
                          Coef. Std.Err.       z  P>|z|  [0.025  0.975]
Intercept                15.870    4.773   3.325  0.001   6.514  25.226
C(domain)[T.body-parts]  -0.159    0.687  -0.232  0.817  -1.507   1.188
C(domain)[T.colours]     -0.318    0.931  -0.342  0.733  -2.143   1.507
C(domain)[T.foods]       -1.061    0.584  -1.816  0.069  -2.206   0.084
Hi_Read                  -0.013    0.980  -0.013  0.990  -1.933   1.908
Hi_Write                 -0.381    0.864  -0.441  0.659  -2.073   1.311
En_Read                  -0.479    1.894  -0.253  0.801  -4.190   3.233
En_Write                 -0.427    1.800  -0.237  0.812  -3.954   3.100
Group Var             

## RQ10 -- Cross-domain trait consistency

In [32]:
print('=' * 70); print('RQ10: CROSS-DOMAIN CONSISTENCY'); print('=' * 70)

wide_words = df_clusters.pivot_table(index='subject_id', columns='domain', values='n_words')
wide_irt   = df_clusters.pivot_table(index='subject_id', columns='domain', values='mean_irt')
print('\nn_words wide table (head):'); print(wide_words.head())

print('\nPairwise Pearson correlations of n_words:')
print(wide_words.corr(method='pearson').round(3))
print('\nPairwise Pearson correlations of mean_irt:')
print(wide_irt.corr(method='pearson').round(3))

# Test each pair
print('\nPair-wise tests (n_words):')
doms = list(wide_words.columns)
for d1, d2 in combinations(doms, 2):
    pair = wide_words[[d1, d2]].dropna()
    if len(pair) >= 5:
        r, p = pearsonr(pair[d1], pair[d2])
        print(f'  {d1:12s} ↔ {d2:12s}  r={r:.3f}  p={p:.4g}  n={len(pair)}')
print('\nPair-wise tests (mean_irt):')
for d1, d2 in combinations(doms, 2):
    pair = wide_irt[[d1, d2]].dropna()
    if len(pair) >= 5:
        r, p = pearsonr(pair[d1], pair[d2])
        print(f'  {d1:12s} ↔ {d2:12s}  r={r:.3f}  p={p:.4g}  n={len(pair)}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(wide_words.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('A. Cross-domain corr: n_words')
sns.heatmap(wide_irt.corr(method='pearson'), annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('B. Cross-domain corr: mean_irt')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ10_cross_domain.png', bbox_inches='tight')
plt.close()
print('Saved: RQ10_cross_domain.png')


RQ10: CROSS-DOMAIN CONSISTENCY

n_words wide table (head):
domain      animals  body-parts  colours  foods
subject_id                                     
3342            7.0         7.0      NaN    7.0
5157           10.0        12.0      NaN    6.0
10147           4.0         6.0      NaN    7.0
10255          21.0         NaN     14.0   15.0
13498           7.0         6.0      NaN    7.0

Pairwise Pearson correlations of n_words:
domain      animals  body-parts  colours  foods
domain                                         
animals       1.000       0.815    0.838  0.581
body-parts    0.815       1.000      NaN  0.510
colours       0.838         NaN    1.000  0.511
foods         0.581       0.510    0.511  1.000

Pairwise Pearson correlations of mean_irt:
domain      animals  body-parts  colours  foods
domain                                         
animals       1.000       0.785    0.872  0.751
body-parts    0.785       1.000      NaN  0.736
colours       0.872         NaN    1.0

## RQ11 -- SpAM ↔ VFT order alignment

In [33]:
print('=' * 70); print('RQ11: SPAM <-> VFT ALIGNMENT'); print('=' * 70)

alignment_rows = []
for _, vft_row in df_vft.iterrows():
    subj, dom = vft_row['subject_id'], vft_row['domain']
    words = vft_row['words']
    spam_match = df_spam[(df_spam['subject_id']==subj) & (df_spam['domain']==dom)]
    if spam_match.empty: continue
    coords = {wc['word']: (wc['x'], wc['y']) for wc in spam_match.iloc[0]['word_coords']}

    # VFT words that exist in SpAM
    seq = [w for w in words if w in coords]
    if len(seq) < 3: continue

    # SpAM-walk: start at seq[0], greedily nearest-neighbor
    visited = [seq[0]]; remaining = set(seq[1:])
    while remaining:
        last = visited[-1]; lx, ly = coords[last]
        nxt = min(remaining, key=lambda w: (coords[w][0]-lx)**2 + (coords[w][1]-ly)**2)
        visited.append(nxt); remaining.discard(nxt)

    # Spearman rank corr between actual VFT order and SpAM-walk order
    spam_pos = {w: i for i, w in enumerate(visited)}
    actual_pos = {w: i for i, w in enumerate(seq)}
    actual = [actual_pos[w] for w in seq]
    spam_order = [spam_pos[w] for w in seq]
    if len(set(spam_order)) > 1 and len(set(actual)) > 1:
        rho, p = spearmanr(actual, spam_order)
        alignment_rows.append({'subject_id':subj,'domain':dom,'rho':rho,
                                'pval':p, 'n_words': len(seq)})

df_align = pd.DataFrame(alignment_rows)
df_align = df_align.merge(df_subjects[['subject_id','hi_fluency']], on='subject_id', how='left')
print(f'\nAlignment trials: {len(df_align)}')
print(df_align.groupby('domain')['rho'].describe().round(3))

# Test: is rho > 0?
for dom in sorted(df_align['domain'].unique()):
    rhos = df_align[df_align['domain']==dom]['rho'].dropna()
    if len(rhos) > 3:
        try:
            wstat, wp = wilcoxon(rhos, alternative='greater')
            print(f'  {dom:12s} mean rho={rhos.mean():.3f} (n={len(rhos)})  W={wstat:.0f}  p={wp:.4g}')
        except Exception as e:
            print(f'  {dom}: failed ({e})')

# rho ~ hi_fluency
sub = df_align.dropna(subset=['rho','hi_fluency'])
if len(sub) > 5:
    r, p = pearsonr(sub['hi_fluency'], sub['rho'])
    print(f'\ncorr(hi_fluency, rho): r={r:.3f}  p={p:.4g}  n={len(sub)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df_align, x='domain', y='rho', hue='domain',
            palette=PALETTE[:df_align['domain'].nunique()], ax=axes[0], legend=False)
axes[0].axhline(0, color='grey', linestyle='--')
axes[0].set_title('A. Spearman rho per domain (VFT order vs SpAM walk)')
axes[1].scatter(sub['hi_fluency'], sub['rho'], color=PALETTE[0], alpha=0.6)
axes[1].axhline(0, color='grey', linestyle='--')
axes[1].set_xlabel('hi_fluency'); axes[1].set_ylabel('rho')
axes[1].set_title('B. Alignment rho vs hi_fluency')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ11_alignment.png', bbox_inches='tight')
plt.close()
print('Saved: RQ11_alignment.png')


RQ11: SPAM <-> VFT ALIGNMENT

Alignment trials: 98
            count   mean    std    min    25%    50%    75%    max
domain                                                            
animals      33.0  0.469  0.326 -0.109  0.200  0.488  0.750  0.964
body-parts   22.0  0.564  0.315  0.024  0.311  0.583  0.821  1.000
colours      11.0  0.294  0.366 -0.534  0.164  0.433  0.545  0.709
foods        32.0  0.554  0.303 -0.191  0.382  0.576  0.736  1.000
  animals      mean rho=0.469 (n=33)  W=547  p=9.59e-07
  body-parts   mean rho=0.564 (n=22)  W=253  p=2.384e-07
  colours      mean rho=0.294 (n=11)  W=56  p=0.021
  foods        mean rho=0.554 (n=32)  W=524  p=5.812e-07

corr(hi_fluency, rho): r=0.215  p=0.03346  n=98
Saved: RQ11_alignment.png


## RQ12 (exploratory) -- Strategy text mining

In [34]:
print('=' * 70); print('RQ12: STRATEGY TEXT MINING (EXPLORATORY)'); print('=' * 70)

if 'strategies' not in df_subjects.columns:
    print('strategies missing -- skipping')
else:
    BUCKETS = {
        'taxonomic': ['domestic','wild','category','sub','type','kind','classify','categor','group'],
        'spatial':   ['layout','room','bedroom','position','spatial','imagery','visual','place','top','bottom'],
        'mnemonic':  ['memor','memorize','memori','remember','recall','sequence','order','linking'],
        'none':      ['none','no strategy','randomly','random','no','nothing'],
    }

    def bucket(text):
        if not isinstance(text, str): return 'unspecified'
        t = text.lower()
        hits = {k: sum(kw in t for kw in kws) for k, kws in BUCKETS.items()}
        if max(hits.values()) == 0: return 'unspecified'
        return max(hits, key=hits.get)

    df_subjects['strategy_bucket'] = df_subjects['strategies'].apply(bucket)
    print('\nStrategy bucket counts:')
    print(df_subjects['strategy_bucket'].value_counts())

    # Test if word_count differs by bucket
    sub = df_clusters.merge(df_subjects[['subject_id','strategy_bucket']], on='subject_id')
    sub = sub[sub['strategy_bucket'] != 'unspecified']
    print('\nMean n_words per strategy bucket:')
    print(sub.groupby('strategy_bucket')['n_words'].agg(['mean','std','count']).round(2))
    if sub['strategy_bucket'].nunique() >= 2:
        groups = [g['n_words'].dropna().values for _, g in sub.groupby('strategy_bucket')]
        if all(len(g) > 0 for g in groups):
            try:
                h, p = kruskal(*groups)
                print(f'  KW H={h:.2f}  p={p:.4g}')
            except Exception as e:
                print(f'  KW failed: {e}')

    fig, ax = plt.subplots(figsize=(8, 4))
    sub['strategy_bucket'].fillna('NA').value_counts().plot(
        kind='bar', ax=ax, color=PALETTE[0], edgecolor='#333')
    ax.set_title('Strategy buckets (n=20 filled / 35)')
    ax.set_ylabel('# subjects'); ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/RQ12_strategies.png', bbox_inches='tight')
    plt.close()
    print('Saved: RQ12_strategies.png')


RQ12: STRATEGY TEXT MINING (EXPLORATORY)

Strategy bucket counts:
strategy_bucket
unspecified    23
taxonomic       6
mnemonic        3
spatial         2
none            1
Name: count, dtype: int64

Mean n_words per strategy bucket:
                  mean   std  count
strategy_bucket                    
mnemonic         12.78  6.48      9
none              5.67  1.53      3
spatial           6.33  1.15      3
taxonomic        10.39  3.38     18
  KW H=8.22  p=0.04175
Saved: RQ12_strategies.png


## Master summary

In [35]:
print('=' * 70); print('MASTER SUMMARY'); print('=' * 70)

findings = []

def safe(p, b=None):
    if isinstance(b, float) and not np.isnan(b):
        return f'beta={b:.3f}, p={p:.3g}'
    return f'p={p:.3g}'

# RQ1
if not np.isnan(rq1_pval):
    findings.append({'RQ':'RQ1','Hypothesis':'Phon. similarity over position',
                     'Result':safe(rq1_pval, rq1_coef),
                     'Supported': 'yes' if rq1_pval<0.05 else 'no'})
findings.append({'RQ':'RQ1','Hypothesis':'Phon. sim higher within clusters',
                 'Result':f'U={u_stat:.0f}, p={u_pval:.3g}',
                 'Supported': 'yes' if u_pval<0.05 else 'no'})

# RQ2
if m1 is not None:
    findings.append({'RQ':'RQ2','Hypothesis':'SpAM dist -> IRT (LMM)',
                     'Result':safe(m1.pvalues['spam_dist_z'], m1.params['spam_dist_z']),
                     'Supported': 'yes' if m1.pvalues['spam_dist_z']<0.05 else 'no'})
if m1 is not None and m2 is not None:
    a1 = aic_bic(m1)[0]; a2 = aic_bic(m2)[0]
    findings.append({'RQ':'RQ2','Hypothesis':'M2 (joint) better than M1',
                     'Result':f'AIC M1={a1:.0f} vs M2={a2:.0f}',
                     'Supported': 'yes' if a2<a1 else 'no'})

# RQ3
for _, row in kw_df.iterrows():
    findings.append({'RQ':'RQ3','Hypothesis':f'Domain diff: {row["Variable"]}',
                     'Result':f'H={row["H"]:.2f}, p={row["p"]:.3g}',
                     'Supported':row['Significant']})

# RQ4
if glm_b is not None:
    bp_term = [t for t in glm_b.params.index if 'fluency' in t and 'body-parts' in t]
    if bp_term:
        findings.append({'RQ':'RQ4','Hypothesis':'fluency x body-parts interaction',
                         'Result':safe(glm_b.pvalues[bp_term[0]], glm_b.params[bp_term[0]]),
                         'Supported': 'yes' if glm_b.pvalues[bp_term[0]]<0.05 else 'no'})
if glm_decomp is not None:
    for term in glm_decomp.params.index:
        if 'Hi_Read' in term or 'Hi_Write' in term:
            findings.append({'RQ':'RQ4','Hypothesis':f'decomp {term}',
                             'Result':safe(glm_decomp.pvalues[term], glm_decomp.params[term]),
                             'Supported': 'yes' if glm_decomp.pvalues[term]<0.05 else 'no'})

# RQ5
if m_dev is not None:
    for term in ['C(l1_hindi)[T.Other]','Hi_Read','Hi_Write','En_Read','En_Write']:
        if term in m_dev.params.index:
            findings.append({'RQ':'RQ5','Hypothesis':f'pct_devanagari ~ {term}',
                             'Result':safe(m_dev.pvalues[term], m_dev.params[term]),
                             'Supported': 'yes' if m_dev.pvalues[term]<0.05 else 'no'})

# RQ6
if m_all is not None:
    for c in [col for col in m_all.params.index if col.startswith('embed_')]:
        findings.append({'RQ':'RQ6','Hypothesis':f'{c} predicts IRT (joint)',
                         'Result':safe(m_all.pvalues[c], m_all.params[c]),
                         'Supported': 'yes' if m_all.pvalues[c]<0.05 else 'no'})

# RQ7 (MVT vs SpAM agreement)
if not np.isnan(p_chi):
    findings.append({'RQ':'RQ7','Hypothesis':'MVT vs SpAM-median agreement (chi2)',
                     'Result':f'chi2={chi2:.2f}, p={p_chi:.3g}',
                     'Supported': 'yes' if p_chi<0.05 else 'no'})

# RQ8
if m_freq is not None:
    findings.append({'RQ':'RQ8','Hypothesis':'curr_freq_log -> log_irt',
                     'Result':safe(m_freq.pvalues['curr_freq_log'], m_freq.params['curr_freq_log']),
                     'Supported': 'yes' if m_freq.pvalues['curr_freq_log']<0.05 else 'no'})

# Build table
sum_df = pd.DataFrame(findings)
print(sum_df.to_string(index=False))
sum_df.to_csv(f'{OUTPUT_DIR}/table_master_findings.csv', index=False)
if not comp_df.empty:
    comp_df.to_csv(f'{OUTPUT_DIR}/table_rq2_modelcomp.csv', index=False)
if not kw_df.empty:
    kw_df.to_csv(f'{OUTPUT_DIR}/table_rq3_kw.csv', index=False)
print(f'\nSaved master table -> {OUTPUT_DIR}/table_master_findings.csv')


MASTER SUMMARY
 RQ                                 Hypothesis                  Result Supported
RQ1             Phon. similarity over position    beta=-0.009, p=0.583        no
RQ1           Phon. sim higher within clusters         U=87319, p=0.33        no
RQ2                     SpAM dist -> IRT (LMM)  beta=0.097, p=5.63e-08       yes
RQ2                  M2 (joint) better than M1  AIC M1=1585 vs M2=1521       yes
RQ3             Domain diff: mean_cluster_size         H=2.80, p=0.423        ns
RQ3                       Domain diff: n_words       H=11.09, p=0.0112         *
RQ3                      Domain diff: mean_irt        H=8.44, p=0.0378         *
RQ3                 Domain diff: mean_phon_sim      H=15.76, p=0.00127        **
RQ3                    Domain diff: n_switches     H=17.28, p=0.000619       ***
RQ4           fluency x body-parts interaction   beta=0.219, p=0.00322       yes
RQ4                           decomp Hi_Read_c     beta=0.069, p=0.517        no
RQ4   decomp 

## Caveats & generated files

In [36]:
print('=' * 70); print('FILES GENERATED'); print('=' * 70)
for f in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(full):
        sz = os.path.getsize(full)
        print(f'  {f:50s} ({sz/1024:.1f} KB)')

print('\n' + '=' * 70); print('CAVEATS'); print('=' * 70)
print('''
- N=35 limits power especially for individual-difference splits.
- Each subject did 3 of 4 domains (animals/foods universal; body-parts/colours alternated)
  so domain x subject is not a balanced design.
- Gender is 32M / 3F -> RQ9 gender effects must be reported as exploratory only.
- Hinglish detection uses heuristics + small per-domain English shortlists; some
  uncommon English words could be miscoded as Hinglish or vice versa.
- Phonological similarity = Levenshtein on ITRANS transliteration, which is closer
  to orthographic than true phonemic similarity.
- MVT switch operationalisation (Hills/Jones/Todd 2012) compares local rate to
  cumulative-mean rate; alternative formulations exist.
- Embedding distances (RQ6) are produced by general-purpose multilingual / Indian
  language models -- not VFT-domain-tuned, so absolute magnitudes should not be
  over-interpreted; relative ordering is the inference target.
- Strategy bucketing (RQ12) uses keyword heuristics on n=20 free-text answers --
  exploratory only.
''')


FILES GENERATED
  EDA_A_script_composition.png                       (107.4 KB)
  EDA_B_demographics.png                             (375.9 KB)
  EDA_D_irt_distribution.png                         (104.3 KB)
  EDA_E_spam_wordcount.png                           (69.5 KB)
  RQ10_cross_domain.png                              (195.9 KB)
  RQ11_alignment.png                                 (221.9 KB)
  RQ12_strategies.png                                (103.9 KB)
  RQ1_phon_over_position.png                         (211.4 KB)
  RQ2_joint_cue.png                                  (71.7 KB)
  RQ3_domain_differences.png                         (469.4 KB)
  RQ4_glm.png                                        (90.3 KB)
  RQ5_script_hinglish.png                            (202.3 KB)
  RQ6_embeddings.png                                 (262.2 KB)
  RQ7_mvt.png                                        (260.2 KB)
  RQ8_typicality.png                                 (271.4 KB)
  RQ9_individual_diffs.png 

In [42]:
!zip -r /content/phase2_outputs_complete.zip /content/phase2_outputs_complete/

  adding: content/phase2_outputs_complete/ (stored 0%)
  adding: content/phase2_outputs_complete/RQ5_script_hinglish.png (deflated 21%)
  adding: content/phase2_outputs_complete/dataframes/ (stored 0%)
  adding: content/phase2_outputs_complete/dataframes/clusters.csv (deflated 75%)
  adding: content/phase2_outputs_complete/dataframes/subjects.csv (deflated 68%)
  adding: content/phase2_outputs_complete/dataframes/words.csv (deflated 78%)
  adding: content/phase2_outputs_complete/dataframes/transitions.csv (deflated 77%)
  adding: content/phase2_outputs_complete/RQ12_strategies.png (deflated 21%)
  adding: content/phase2_outputs_complete/EDA_D_irt_distribution.png (deflated 25%)
  adding: content/phase2_outputs_complete/RQ6_embeddings.png (deflated 15%)
  adding: content/phase2_outputs_complete/embeddings_cache.pkl (deflated 9%)
  adding: content/phase2_outputs_complete/RQ2_joint_cue.png (deflated 33%)
  adding: content/phase2_outputs_complete/RQ3_domain_differences.png (deflated 17%)
 

In [41]:
# === POSTER FIGURES ===
# Panel 3 (RQ5): Mean IRT by transition class with significance
import matplotlib.patches as mpatches
order = ['pure_english', 'mixed', 'hinglish', 'pure_devanagari']
labels = ['English\n→English', 'Mixed\nscript', 'Hinglish\n→Hinglish', 'Devanagari\n→Devanagari']
means = [df_trans[df_trans['trans_class']==c]['irt_ms'].mean() for c in order]
sems  = [df_trans[df_trans['trans_class']==c]['irt_ms'].sem()*1.96 for c in order]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=sems, color=[PALETTE[0], PALETTE[2], PALETTE[1], PALETTE[3]],
              edgecolor='#333', capsize=5)
ymax = max(m + s for m, s in zip(means, sems))
offset = ymax * 0.04
for b, m, s in zip(bars, means, sems):
    ax.text(b.get_x()+b.get_width()/2, m + s + offset, f'{m:.0f} ms',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean IRT (ms)')
ax.set_title('Devanagari typing penalty: 2.2× slower than English\n(all pairwise p < 10⁻⁵)')
ax.set_ylim(0, ymax * 1.18)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/POSTER_RQ5_script_irt.png', bbox_inches='tight', dpi=300)
plt.close()

# Panel 4 (RQ7): Pre vs post-switch IRT (MVT spike test)
pre, post = [], []
for (subj, dom), grp in df_trans.groupby(['subject_id', 'domain']):
    g = grp.sort_values('position').reset_index(drop=True)
    for i, row in g.iterrows():
        if row['is_switch_mvt'] == 1 and 0 < i < len(g)-1:
            pre.append(g.iloc[i-1]['irt_ms']); post.append(g.iloc[i+1]['irt_ms'])
pre, post = np.array(pre), np.array(post)
fig, ax = plt.subplots(figsize=(6, 5))
labels = ['Pre-switch\n(IRT_{i-1})', 'Post-switch\n(IRT_{i+1})']
means = [pre.mean(), post.mean()]
sems  = [pre.std()/np.sqrt(len(pre))*1.96, post.std()/np.sqrt(len(post))*1.96]
bars = ax.bar(labels, means, yerr=sems, color=[PALETTE[0], PALETTE[3]],
              edgecolor='#333', capsize=8, width=0.5)
ymax_err = max(m + s for m, s in zip(means, sems))
offset = ymax_err * 0.04
for b, m, s in zip(bars, means, sems):
    ax.text(b.get_x()+b.get_width()/2, m + s + offset, f'{m:.0f} ms',
            ha='center', fontsize=11, fontweight='bold')
bracket_y = ymax_err * 1.18
ax.plot([0, 1], [bracket_y]*2, 'k-', linewidth=1)
ax.text(0.5, bracket_y * 1.02, 'W=17,640, p=4.2×10⁻⁶', ha='center', fontsize=10)
ax.set_ylabel('Mean IRT (ms)')
ax.set_title('Marginal-value-theorem prediction confirmed:\npost-switch IRT spike')
ax.set_ylim(0, bracket_y * 1.10)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/POSTER_RQ7_mvt_spike.png', bbox_inches='tight', dpi=300)
plt.close()

# Panel 5 (RQ8): log-IRT vs production frequency
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sub = df_trans.dropna(subset=['log_irt', 'curr_freq_log'])
axes[0].scatter(sub['curr_freq_log'], sub['log_irt'], alpha=0.15, s=10, color=PALETTE[0])
z = np.polyfit(sub['curr_freq_log'], sub['log_irt'], 1)
xr = np.linspace(sub['curr_freq_log'].min(), sub['curr_freq_log'].max(), 100)
axes[0].plot(xr, np.polyval(z, xr), color=PALETTE[1], linewidth=2.5)
axes[0].set_xlabel('log(1 + production frequency)')
axes[0].set_ylabel('log(1 + IRT ms)')
axes[0].set_title('A. Typical words retrieved faster\n(β=−0.124, p=0.0005)')
# Right panel: typicality vs fluency by domain (existing)
dom_subj = df_words.dropna(subset=['hi_fluency']).groupby(['subject_id','domain']).agg(
    mean_typ=('prod_freq_pct','mean'), hi_fluency=('hi_fluency','first')).reset_index()
sns.scatterplot(data=dom_subj, x='hi_fluency', y='mean_typ', hue='domain',
                palette=PALETTE[:dom_subj['domain'].nunique()], ax=axes[1], s=60)
axes[1].set_xlabel('Hindi fluency (Likert)'); axes[1].set_ylabel('Mean typicality (% subjects)')
axes[1].set_title('B. High-fluency → more typical words\n(body-parts r=+0.60, p=0.002)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/POSTER_RQ8_typicality.png', bbox_inches='tight', dpi=300)
plt.close()

print('Saved 3 poster figures: POSTER_RQ5_script_irt.png, POSTER_RQ7_mvt_spike.png, POSTER_RQ8_typicality.png')


Saved 3 poster figures: POSTER_RQ5_script_irt.png, POSTER_RQ7_mvt_spike.png, POSTER_RQ8_typicality.png
